In [ ]:
import os
os.chdir(r'D:\HKUST\5054_Statistical_Machine_Learning\Assignments\HW1')
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import summary_table
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Life Expectancy Data.csv')
print(f"drop colums")
df = df.drop(columns=['Country'], errors='ignore')

df['Status'] = df['Status'].map({'Developing': 0, 'Developed': 1})

print(df.columns.tolist()) 
print(f"\nMissing values number before dropping:")
print(df.isnull().sum())
df = df.dropna()
print(f"Shape of Dataframe after dropping NA: {df.shape}")
predictors = [col for col in df.columns if col != 'Life expectancy '] 
X = df[predictors]
y = df['Life expectancy ']
X = sm.add_constant(X)
full_model = sm.OLS(y, X).fit()

print(f"\nQuestion 1: Full Model Summary")
print(full_model.summary())
significant_vars = full_model.pvalues[full_model.pvalues < 0.05].index.tolist()
print(f"\nSignificant predicting variables (p < 0.05): {significant_vars}")


print(f"\nQuestion 2: 95% CI for 'Adult Mortality' and 'HIV/AIDS'")
conf_int_95 = full_model.conf_int(alpha=0.05)
adult_mort_ci = conf_int_95.loc['Adult Mortality']
hiv_aids_ci = conf_int_95.loc[' HIV/AIDS'] 
print(f"Adult Mortality 95% CI: [{adult_mort_ci[0]:.4f}, {adult_mort_ci[1]:.4f}]")
print(f"HIV/AIDS 95% CI: [{hiv_aids_ci[0]:.4f}, {hiv_aids_ci[1]:.4f}]")


print("\nQuestion 3: 97% CI for 'Schooling' and 'Alcohol'")
conf_int_97 = full_model.conf_int(alpha=0.03)
schooling_ci = conf_int_97.loc['Schooling']
alcohol_ci = conf_int_97.loc['Alcohol']
print(f"Schooling 97% CI: [{schooling_ci[0]:.4f}, {schooling_ci[1]:.4f}]")
print(f"Alcohol 97% CI: [{alcohol_ci[0]:.4f}, {alcohol_ci[1]:.4f}]")


print("\nQuestion 4: Top 7 predictors by p-value")
p_values = full_model.pvalues.drop('const').sort_values()
top_7_predictors = p_values.head(7).index.tolist()
print(f"Top 7 most influential predictors by p-value: {top_7_predictors}")
X_small = df[top_7_predictors]
X_small = sm.add_constant(X_small)
small_model = sm.OLS(y, X_small).fit()
print(small_model.summary())
print(small_model.conf_int())
print(small_model.pvalues)
print(small_model.rsquared)
print(small_model.rsquared_adj)
print(small_model.tvalues)
print(small_model.fvalue)
print(small_model.mse_resid)
print(small_model.bse)


print("\nQuestion 5: Prediction with 99% CI")
new_obs_data = {
    'Year': [2008],
    'Status': [1], # Developed
    'Adult Mortality': [125],
    'infant deaths': [94],
    'Alcohol': [4.1],
    'percentage expenditure': [100],
    'Hepatitis B': [20],
    'Measles': [13],
    'BMI': [55],
    'under-five deaths ': [2],
    'Polio': [12],
    'Total expenditure': [5.9],
    'Diphtheria': [12],
    ' HIV/AIDS': [0.5],
    'GDP': [5892],
    'Population': [1.34e6],
    'thinness  1-19 years': [0],
    'thinness 5-9 years': [0],   
    'Income composition of resources': [0.9],
    'Schooling': [18]
}

new_obs = pd.DataFrame(new_obs_data)
new_obs_small = new_obs[top_7_predictors] 
new_obs_small = sm.add_constant(new_obs_small, has_constant='add') 
prediction = small_model.get_prediction(new_obs_small)
pred_summary = prediction.summary_frame(alpha=0.01)
print(pred_summary.head())
print(f"Predicted Life Expectancy: {pred_summary['mean'].iloc[0]:.2f}")
print(f"99% Confidence Interval: [{pred_summary['mean_ci_lower'].iloc[0]:.2f}, {pred_summary['mean_ci_upper'].iloc[0]:.2f}]")

print("\n Question 6: AIC Comparison")
print(f"AIC of Full Model: {full_model.aic:.2f}")
print(f"AIC of Smaller Model: {small_model.aic:.2f}")
print(f"BIC of Full Model: {full_model.bic:.2f}")
print(f"BIC of Smaller Model: {small_model.bic:.2f}")
print(f"Adjusted R-squared of Full Model: {full_model.rsquared_adj:.4f}")
print(f"Adjusted R-squared of Smaller Model: {small_model.rsquared_adj:.4f}")

n = len(y)
p_full = full_model.params.shape[0]   
p_small = small_model.params.shape[0]
sigma2_full = full_model.mse_resid   
Cp_full = full_model.ssr / sigma2_full - (n - 2 * p_full)
Cp_small = small_model.ssr / sigma2_full - (n - 2 * p_small)
print("\nMallows' Cp:")
print(f"Cp (Full model): {Cp_full:.4f}")
print(f"Cp (Small model): {Cp_small:.4f}")

from sklearn.model_selection import KFold
def cv_mse(columns, X_df, y_series, k=5, random_state=42):
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    mses = []
    for train_idx, test_idx in kf.split(X_df):
        X_train = X_df.iloc[train_idx][columns]
        X_test = X_df.iloc[test_idx][columns]
        y_train = y_series.iloc[train_idx]
        y_test = y_series.iloc[test_idx]
        X_train_c = sm.add_constant(X_train, has_constant='add')
        X_test_c = sm.add_constant(X_test, has_constant='add')
        mdl = sm.OLS(y_train, X_train_c).fit()
        preds = mdl.predict(X_test_c)
        mses.append(np.mean((y_test - preds) ** 2))
    return np.mean(mses), np.std(mses)
full_columns = predictors  
mse_full_mean, mse_full_std = cv_mse(full_columns, df, y, k=5)
mse_small_mean, mse_small_std = cv_mse(top_7_predictors, df, y, k=5)
print("\n5-fold CV MSE:")
print(f"Full model CV MSE: {mse_full_mean:.4f} ± {mse_full_std:.4f}")
print(f"Small model CV MSE: {mse_small_mean:.4f} ± {mse_small_std:.4f}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import statsmodels.api as sm
import warnings
df = pd.read_csv('FirstYearGPA.csv')
expected_columns = ['GPA', 'HSGPA', 'SATV', 'SATM', 'Male', 'HU', 'SS', 'FirstGen', 'White', 'CollegeBound']
if not set(expected_columns).issubset(set(df.columns)):
    raise ValueError("数据列名不匹配，请检查列名是否为: " + str(expected_columns))

df['good_position'] = (df['GPA'] >= 3.0).astype(int)

print("数据形状:", df.shape)
print("前5行:")
print(df.head())

from itertools import combinations
X = df.drop(['GPA', 'good_position'], axis=1)  
y = df['GPA']  

best_models = {}
r_squared_adj = []

for k in range(1, 9):  # k从1到8
    best_r2_adj = -np.inf
    best_combination = None
    best_model = None
    
    for comb in combinations(X.columns, k):
        X_subset = X[list(comb)]
        X_subset_sm = sm.add_constant(X_subset) 
        
        try:
            model = sm.OLS(y, X_subset_sm).fit()
            r2_adj = model.rsquared_adj
            
            if r2_adj > best_r2_adj:
                best_r2_adj = r2_adj
                best_combination = comb
                best_model = model
        except:
            continue 
    
    best_models[k] = {
        'features': best_combination,
        'model': best_model,
        'r2_adj': best_r2_adj
    }
    r_squared_adj.append(best_r2_adj)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 9), r_squared_adj, marker='o', linestyle='-', color='b')
plt.title('Adjusted R-squared vs Model Size (k)')
plt.xlabel('Number of Features (k)')
plt.ylabel('Adjusted R-squared')
plt.grid(True)
plt.xticks(range(1, 9))
plt.show()

best_k = np.argmax(r_squared_adj) + 1
best_model_info = best_models[best_k]
print(f"\n最佳模型 (k={best_k}):")
print("特征:", best_model_info['features'])
print("调整R²:", best_model_info['r2_adj'])
print("\n模型摘要:")
print(best_model_info['model'].summary())


print(f"CV choose best subset")
from sklearn.model_selection import cross_validate
def get_best_subset_cv(X, y, k_list=[1,2,3,4,5,6,7,8]):
    results = {}
    for k in k_list:
        best_score = -np.inf
        best_features = None
        for comb in combinations(X.columns, k):
            X_subset = X[list(comb)]
            model = LinearRegression()
            cv_scores = cross_val_score(model, X_subset, y, cv=5, scoring='r2')
            mean_score = np.mean(cv_scores)
            if mean_score > best_score:
                best_score = mean_score
                best_features = comb
        results[k] = {'features': best_features, 'mean_cv_r2': best_score}
    return results

cv_results = get_best_subset_cv(X, y)

best_k_cv = max(cv_results.keys(), key=lambda k: cv_results[k]['mean_cv_r2'])
best_model_cv = cv_results[best_k_cv]

print(f" (k={best_k_cv}):")
print("特征:", best_model_cv['features'])
print("平均CV R²:", best_model_cv['mean_cv_r2'])

X_best = X[list(best_model_cv['features'])]
X_best_sm = sm.add_constant(X_best)
model_final = sm.OLS(y, X_best_sm).fit()
print(model_final.summary())


def forward_stepwise_bic(X, y, max_features=8):
    included = set()
    bic_values = []
    
    for k in range(1, max_features + 1):
        best_bic = np.inf
        best_feature = None
        
        for feature in X.columns:
            if feature in included:
                continue
            candidate_features = list(included) + [feature]
            X_candidate = X[candidate_features]
            X_candidate_sm = sm.add_constant(X_candidate)
            try:
                model = sm.OLS(y, X_candidate_sm).fit()
                bic = model.bic
                if bic < best_bic:
                    best_bic = bic
                    best_feature = feature
            except:
                continue
        
        if best_feature is not None:
            included.add(best_feature)
            bic_values.append(best_bic)
        else:
            break 
    
    steps = []
    current_features = []
    for i, feat in enumerate(sorted(included)):
        current_features.append(feat)
        X_temp = X[current_features]
        X_temp_sm = sm.add_constant(X_temp)
        model = sm.OLS(y, X_temp_sm).fit()
        steps.append({
            'k': i+1,
            'features': current_features.copy(),
            'bic': model.bic,
            'model': model
        })
    
    return steps

forward_steps = forward_stepwise_bic(X, y, max_features=8)

bic_list = [step['bic'] for step in forward_steps]

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(bic_list)+1), bic_list, marker='o', linestyle='-', color='g')
plt.title('BIC vs Model Size (Forward Stepwise)')
plt.xlabel('Number of Features (k)')
plt.ylabel('BIC')
plt.grid(True)
plt.xticks(range(1, len(bic_list)+1))
plt.show()

best_step = min(forward_steps, key=lambda x: x['bic'])
best_k_bic = best_step['k']
print(f"\n前向逐步选择最佳模型 (k={best_k_bic}, BIC={best_step['bic']:.4f}):")
print("特征:", best_step['features'])
print("\n模型摘要:")
print(best_step['model'].summary())



def forward_stepwise_cv(X, y, max_features=8, cv=5):
    included = set()
    cv_scores = []
    
    for k in range(1, max_features + 1):
        best_score = -np.inf
        best_feature = None
        
        for feature in X.columns:
            if feature in included:
                continue
            candidate_features = list(included) + [feature]
            X_candidate = X[candidate_features]
            model = LinearRegression()
            cv_scores_fold = cross_val_score(model, X_candidate, y, cv=cv, scoring='r2')
            mean_score = np.mean(cv_scores_fold)
            if mean_score > best_score:
                best_score = mean_score
                best_feature = feature
        
        if best_feature is not None:
            included.add(best_feature)
            cv_scores.append(best_score)
        else:
            break
    
    steps = []
    current_features = []
    for i, feat in enumerate(sorted(included)):
        current_features.append(feat)
        X_temp = X[current_features]
        model = LinearRegression()
        cv_scores_fold = cross_val_score(model, X_temp, y, cv=cv, scoring='r2')
        mean_score = np.mean(cv_scores_fold)
        steps.append({
            'k': i+1,
            'features': current_features.copy(),
            'mean_cv_r2': mean_score
        })
    
    return steps

forward_cv_steps = forward_stepwise_cv(X, y, max_features=8, cv=5)

# 绘图：CV R² vs k
cv_r2_list = [step['mean_cv_r2'] for step in forward_cv_steps]
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cv_r2_list)+1), cv_r2_list, marker='o', linestyle='-', color='orange')
plt.title('Mean CV R² vs Model Size (Forward Stepwise + 5-fold CV)')
plt.xlabel('Number of Features (k)')
plt.ylabel('Mean CV R²')
plt.grid(True)
plt.xticks(range(1, len(cv_r2_list)+1))
plt.show()

best_step_cv = max(forward_cv_steps, key=lambda x: x['mean_cv_r2'])
best_k_cv_forward = best_step_cv['k']
print(f"\n前向逐步 + 5折CV 最佳模型 (k={best_k_cv_forward}):")
print("特征:", best_step_cv['features'])
print("平均CV R²:", best_step_cv['mean_cv_r2'])

X_best_forward = X[best_step_cv['features']]
X_best_forward_sm = sm.add_constant(X_best_forward)
model_final_forward = sm.OLS(y, X_best_forward_sm).fit()
print("\n模型摘要:")
print(model_final_forward.summary())


X_log = X 
y_log = df['good_position']

def forward_stepwise_logistic_cv(X, y, max_features=8, cv=5):
    included = set()
    cv_scores = []
    
    for k in range(1, max_features + 1):
        best_score = -np.inf
        best_feature = None
        
        for feature in X.columns:
            if feature in included:
                continue
            candidate_features = list(included) + [feature]
            X_candidate = X[candidate_features]
            model = LogisticRegression(random_state=42, max_iter=1000)
            cv_scores_fold = cross_val_score(model, X_candidate, y, cv=cv, scoring='accuracy')
            mean_score = np.mean(cv_scores_fold)
            if mean_score > best_score:
                best_score = mean_score
                best_feature = feature
        
        if best_feature is not None:
            included.add(best_feature)
            cv_scores.append(best_score)
        else:
            break
    
    steps = []
    current_features = []
    for i, feat in enumerate(sorted(included)):
        current_features.append(feat)
        X_temp = X[current_features]
        model = LogisticRegression(random_state=42, max_iter=1000)
        cv_scores_fold = cross_val_score(model, X_temp, y, cv=cv, scoring='accuracy')
        mean_score = np.mean(cv_scores_fold)
        steps.append({
            'k': i+1,
            'features': current_features.copy(),
            'mean_cv_accuracy': mean_score
        })
    
    return steps

logistic_cv_steps = forward_stepwise_logistic_cv(X_log, y_log, max_features=8, cv=5)

acc_list = [step['mean_cv_accuracy'] for step in logistic_cv_steps]
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(acc_list)+1), acc_list, marker='o', linestyle='-', color='purple')
plt.title('Mean CV Accuracy vs Model Size (Logistic Regression + Forward Stepwise)')
plt.xlabel('Number of Features (k)')
plt.ylabel('Mean CV Accuracy')
plt.grid(True)
plt.xticks(range(1, len(acc_list)+1))
plt.show()

best_step_log = max(logistic_cv_steps, key=lambda x: x['mean_cv_accuracy'])
best_k_log = best_step_log['k']
print(f"\n【题目5】逻辑回归最佳模型 (k={best_k_log}):")
print("特征:", best_step_log['features'])
print("平均CV准确率:", best_step_log['mean_cv_accuracy'])

X_best_log = X_log[best_step_log['features']]
X_best_log_sm = sm.add_constant(X_best_log)
model_log_final = sm.Logit(y_log, X_best_log_sm).fit(disp=False)
print("\n逻辑回归模型摘要:")
print(model_log_final.summary())

model_sk = LogisticRegression(random_state=42, max_iter=1000)
model_sk.fit(X_best_log, y_log)
y_pred = model_sk.predict(X_best_log)
print("\n训练集分类报告:")
print(classification_report(y_log, y_pred))


from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
def forward_stepwise_lda_cv(X, y, max_features=8, cv=5):
    included = set()
    cv_scores = []
    
    for k in range(1, max_features + 1):
        best_score = -np.inf
        best_feature = None
        
        for feature in X.columns:
            if feature in included:
                continue
            candidate_features = list(included) + [feature]
            X_candidate = X[candidate_features]
            model = LinearDiscriminantAnalysis()
            cv_scores_fold = cross_val_score(model, X_candidate, y, cv=cv, scoring='accuracy')
            mean_score = np.mean(cv_scores_fold)
            if mean_score > best_score:
                best_score = mean_score
                best_feature = feature
        
        if best_feature is not None:
            included.add(best_feature)
            cv_scores.append(best_score)
        else:
            break
    
    steps = []
    current_features = []
    for i, feat in enumerate(sorted(included)):
        current_features.append(feat)
        X_temp = X[current_features]
        model = LinearDiscriminantAnalysis()
        cv_scores_fold = cross_val_score(model, X_temp, y, cv=cv, scoring='accuracy')
        mean_score = np.mean(cv_scores_fold)
        steps.append({
            'k': i+1,
            'features': current_features.copy(),
            'mean_cv_accuracy': mean_score
        })
    
    return steps

lda_cv_steps = forward_stepwise_lda_cv(X_log, y_log, max_features=8, cv=5)

lda_acc_list = [step['mean_cv_accuracy'] for step in lda_cv_steps]
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(lda_acc_list)+1), lda_acc_list, marker='o', linestyle='-', color='red')
plt.title('Mean CV Accuracy vs Model Size (LDA + Forward Stepwise)')
plt.xlabel('Number of Features (k)')
plt.ylabel('Mean CV Accuracy')
plt.grid(True)
plt.xticks(range(1, len(lda_acc_list)+1))
plt.show()

best_step_lda = max(lda_cv_steps, key=lambda x: x['mean_cv_accuracy'])
best_k_lda = best_step_lda['k']
print(f"\n【题目6】LDA最佳模型 (k={best_k_lda}):")
print("特征:", best_step_lda['features'])
print("平均CV准确率:", best_step_lda['mean_cv_accuracy'])

X_best_lda = X_log[best_step_lda['features']]
model_lda_final = LinearDiscriminantAnalysis()
model_lda_final.fit(X_best_lda, y_log)

print("\nLDA模型信息:")
print("类均值:")
print(model_lda_final.means_)
print("判别系数 (线性判别函数权重):")
print(model_lda_final.coef_)
print("截距:")
print(model_lda_final.intercept_)

y_pred_lda = model_lda_final.predict(X_best_lda)
print("\n训练集分类报告:")
print(classification_report(y_log, y_pred_lda))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# 加载训练集和测试集
train_df = pd.read_csv('diabetes_train.csv')
test_df = pd.read_csv('diabetes_test.csv')

print("训练集形状:", train_df.shape)
print("测试集形状:", test_df.shape)

# 检查列名是否匹配（根据题目描述）
expected_columns = ['Y', 'age', 'sex', 'bmi', 'map', 'tc', 'ldl', 'hdl', 'tch', 'ltg', 'glu', 'age.tch']
if not set(expected_columns).issubset(set(train_df.columns)):
    raise ValueError("训练集列名不匹配，请检查列名是否为: " + str(expected_columns))

# 检查测试集是否包含相同特征列（除Y外）
feature_cols = [col for col in expected_columns if col != 'Y']

print("\n训练集前5行:")
print(train_df.head())

X_train = train_df[feature_cols]
y_train = train_df['Y']

# 标准化特征（LASSO对尺度敏感）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 生成 lambda 序列：10^seq(4, -2, length=100)
lambda_grid = np.logspace(4, -2, 100, base=10)

# 存储每个lambda对应的系数和ℓ₁范数
coefficients = []
l1_norms = []

for lamb in lambda_grid:
    lasso = Lasso(alpha=lamb, max_iter=10000, random_state=42)
    lasso.fit(X_train_scaled, y_train)
    coef = lasso.coef_
    coefficients.append(coef)
    l1_norms.append(np.sum(np.abs(coef)))

coefficients = np.array(coefficients)
l1_norms = np.array(l1_norms)

# 绘制系数路径图
plt.figure(figsize=(12, 8))
for i, feature in enumerate(feature_cols):
    plt.plot(l1_norms, coefficients[:, i], label=feature)

plt.xlabel('ℓ₁ Norm of Coefficients (||β||₁)')
plt.ylabel('Coefficient Value')
plt.title('LASSO Coefficient Paths vs ℓ₁ Norm')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

# 找出哪些变量在某个λ下被保留（非零）
print("\n【题目1】观察发现：")
print("- 随着ℓ₁范数减小（即正则化减弱），更多变量系数从0变为非零。")
print("- 通常，'bmi', 'ltg', 'map', 'glu' 等变量较早进入模型，说明它们对Y影响较大。")
print("- 当ℓ₁范数很小时，所有变量都可能被选入。")

# 使用 GridSearchCV 进行10折交叉验证
lasso_cv = Lasso(max_iter=10000, random_state=42)
param_grid = {'alpha': lambda_grid}

kf = KFold(n_splits=10, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=lasso_cv,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_squared_error',  # 因为sklearn默认最大化得分，MSE越小越好
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

best_lambda = grid_search.best_params_['alpha']
best_lasso = grid_search.best_estimator_

print(f"\n【题目2】最佳lambda值: {best_lambda:.6f}")
print(f"最佳CV MSE: {-grid_search.best_score_:.6f}")

# 获取最终模型系数
coef_final = best_lasso.coef_
nonzero_mask = coef_final != 0
selected_features = np.array(feature_cols)[nonzero_mask]
num_selected = len(selected_features)

print(f"选中的变量数量: {num_selected}")
print("选中的变量:", selected_features.tolist())

# 绘制 CV error vs lambda
cv_results = grid_search.cv_results_
mean_mse = -cv_results['mean_test_score']  # 转换回MSE

plt.figure(figsize=(10, 6))
plt.semilogx(lambda_grid, mean_mse, marker='o', linestyle='-', color='blue')
plt.axvline(best_lambda, color='red', linestyle='--', label=f'Best λ={best_lambda:.2e}')
plt.xlabel('Lambda (α)')
plt.ylabel('Mean Squared Error (CV)')
plt.title('CV Error vs Lambda for LASSO')
plt.legend()
plt.grid(True)
plt.show()

# 对测试集进行标准化（使用训练集的scaler）
X_test = test_df[feature_cols]
X_test_scaled = scaler.transform(X_test)
y_test = test_df['Y']

# 预测
y_pred_test = best_lasso.predict(X_test_scaled)

# 计算测试集MSE
test_mse = mean_squared_error(y_test, y_pred_test)
print(f"\n【题目3】测试集平均平方误差 (MSE): {test_mse:.6f}")

# 可选：绘制预测 vs 实际值
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_test, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Y')
plt.ylabel('Predicted Y')
plt.title('Test Set: Actual vs Predicted (LASSO)')
plt.grid(True)
plt.show()

# 使用选中的变量，在训练集上用OLS拟合（无正则化）
X_train_selected = X_train[selected_features]
X_train_selected_sm = sm.add_constant(X_train_selected)  # 添加常数项

model_ols = sm.OLS(y_train, X_train_selected_sm).fit()
print(f"\n【题目4】基于选中变量的OLS模型95%置信区间:")
print(model_ols.summary())

# 提取置信区间
conf_int = model_ols.conf_int()
print("\n系数95%置信区间:")
for feat, (lower, upper) in zip(['const'] + selected_features.tolist(), conf_int.values):
    print(f"{feat}: [{lower:.4f}, {upper:.4f}]")

# 创建分类目标变量：stable = 1 if Y < 150, else 0
train_df['stable'] = (train_df['Y'] < 150).astype(int)
y_train_class = train_df['stable']

# 重新标准化特征（虽然之前已标准化，但为了清晰重做）
X_train_scaled_class = scaler.fit_transform(X_train)  # 重新拟合scaler？不，应使用训练集的scaler

# 使用L1正则化逻辑回归（即LASSO-logistic）
logistic_cv = LogisticRegression(penalty='l1', solver='liblinear', max_iter=10000, random_state=42)
param_grid_log = {'C': 1 / lambda_grid}  # C = 1/alpha

grid_search_log = GridSearchCV(
    estimator=logistic_cv,
    param_grid=param_grid_log,
    cv=kf,
    scoring='accuracy',  # 也可以用 'neg_log_loss'
    n_jobs=-1
)

grid_search_log.fit(X_train_scaled_class, y_train_class)

best_C_log = grid_search_log.best_params_['C']
best_lambda_log = 1 / best_C_log
best_logistic = grid_search_log.best_estimator_

print(f"\n【题目5】LASSO-logistic 最佳C值: {best_C_log:.6f}")
print(f"对应最佳lambda值: {best_lambda_log:.6f}")
print(f"最佳CV准确率: {grid_search_log.best_score_:.6f}")

# 获取选中变量
coef_log = best_logistic.coef_[0]  # 多分类时注意维度
nonzero_mask_log = coef_log != 0
selected_features_log = np.array(feature_cols)[nonzero_mask_log]
num_selected_log = len(selected_features_log)

print(f"选中的变量数量: {num_selected_log}")
print("选中的变量:", selected_features_log.tolist())

# 绘制 CV error vs lambda (这里用 accuracy，也可用 -log_loss)
cv_results_log = grid_search_log.cv_results_
mean_acc = cv_results_log['mean_test_score']

plt.figure(figsize=(10, 6))
plt.semilogx(1 / np.array(param_grid_log['C']), mean_acc, marker='o', linestyle='-', color='green')
plt.axvline(best_lambda_log, color='red', linestyle='--', label=f'Best λ={best_lambda_log:.2e}')
plt.xlabel('Lambda (α)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('CV Accuracy vs Lambda for LASSO-Logistic')
plt.legend()
plt.grid(True)
plt.show()

# （可选）在测试集上评估分类器
test_df['stable'] = (test_df['Y'] < 150).astype(int)
y_test_class = test_df['stable']
X_test_scaled_class = scaler.transform(X_test)

y_pred_class = best_logistic.predict(X_test_scaled_class)
test_acc = accuracy_score(y_test_class, y_pred_class)
print(f"\n测试集分类准确率: {test_acc:.6f}")
print("\n测试集分类报告:")
print(classification_report(y_test_class, y_pred_class))

# 使用选中的变量，在训练集上用OLS拟合（无正则化）
X_train_selected = X_train[selected_features]
X_train_selected_sm = sm.add_constant(X_train_selected)  # 添加常数项

model_ols = sm.OLS(y_train, X_train_selected_sm).fit()
print(f"\n【题目4】基于选中变量的OLS模型95%置信区间:")
print(model_ols.summary())

# 提取置信区间
conf_int = model_ols.conf_int()
print("\n系数95%置信区间:")
for feat, (lower, upper) in zip(['const'] + selected_features.tolist(), conf_int.values):
    print(f"{feat}: [{lower:.4f}, {upper:.4f}]")

# 创建分类目标变量：stable = 1 if Y < 150, else 0
train_df['stable'] = (train_df['Y'] < 150).astype(int)
y_train_class = train_df['stable']

# 重新标准化特征（虽然之前已标准化，但为了清晰重做）
X_train_scaled_class = scaler.fit_transform(X_train)  # 重新拟合scaler？不，应使用训练集的scaler

# 使用L1正则化逻辑回归（即LASSO-logistic）
logistic_cv = LogisticRegression(penalty='l1', solver='liblinear', max_iter=10000, random_state=42)
param_grid_log = {'C': 1 / lambda_grid}  # C = 1/alpha

grid_search_log = GridSearchCV(
    estimator=logistic_cv,
    param_grid=param_grid_log,
    cv=kf,
    scoring='accuracy',  # 也可以用 'neg_log_loss'
    n_jobs=-1
)

grid_search_log.fit(X_train_scaled_class, y_train_class)

best_C_log = grid_search_log.best_params_['C']
best_lambda_log = 1 / best_C_log
best_logistic = grid_search_log.best_estimator_

print(f"\n【题目5】LASSO-logistic 最佳C值: {best_C_log:.6f}")
print(f"对应最佳lambda值: {best_lambda_log:.6f}")
print(f"最佳CV准确率: {grid_search_log.best_score_:.6f}")

# 获取选中变量
coef_log = best_logistic.coef_[0]  # 多分类时注意维度
nonzero_mask_log = coef_log != 0
selected_features_log = np.array(feature_cols)[nonzero_mask_log]
num_selected_log = len(selected_features_log)

print(f"选中的变量数量: {num_selected_log}")
print("选中的变量:", selected_features_log.tolist())

# 绘制 CV error vs lambda (这里用 accuracy，也可用 -log_loss)
cv_results_log = grid_search_log.cv_results_
mean_acc = cv_results_log['mean_test_score']

plt.figure(figsize=(10, 6))
plt.semilogx(1 / np.array(param_grid_log['C']), mean_acc, marker='o', linestyle='-', color='green')
plt.axvline(best_lambda_log, color='red', linestyle='--', label=f'Best λ={best_lambda_log:.2e}')
plt.xlabel('Lambda (α)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('CV Accuracy vs Lambda for LASSO-Logistic')
plt.legend()
plt.grid(True)
plt.show()

# （可选）在测试集上评估分类器
test_df['stable'] = (test_df['Y'] < 150).astype(int)
y_test_class = test_df['stable']
X_test_scaled_class = scaler.transform(X_test)

y_pred_class = best_logistic.predict(X_test_scaled_class)
test_acc = accuracy_score(y_test_class, y_pred_class)
print(f"\n测试集分类准确率: {test_acc:.6f}")
print("\n测试集分类报告:")
print(classification_report(y_test_class, y_pred_class))

In [ ]:
import os
os.chdir(r'D:\HKUST\5054_Statistical_Machine_Learning\Assignments\HW1')
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import time

train_df = pd.read_csv('boston_housing_train.csv')
test_df = pd.read_csv('boston_housing_test.csv')
print(train_df.head())
print(train_df.columns.tolist)
X_train = train_df.drop(columns=['medv'])
y_train = train_df['medv']
X_test = test_df.drop(columns=['medv'])
y_test = test_df['medv']

class KNNRegressor:
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X_train, y_train):
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)
    
    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def predict_single(self, x):
        distances = [self.euclidean_distance(x, x_train) for x_train in self.X_train]        
        k_indices = np.argsort(distances)[:self.k]        
        k_nearest_labels = self.y_train[k_indices]        
        return np.mean(k_nearest_labels)
    
    def predict(self, X_test):
        X_test = np.array(X_test)
        predictions = [self.predict_single(x) for x in X_test]
        return np.array(predictions)
    
print("Question 1: KNN Regression without Standardization\n")
results = []

for k in range(1, 21): 
    knn = KNNRegressor(k=k)  
    start_time = time.time()
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    end_time = time.time()    
    mse = mean_squared_error(y_test, y_pred)
    running_time = end_time - start_time    
    results.append({
        'K': k,
        'MSE': mse,
        'Running Time (s)': running_time
    })
    
    print(f"K={k:2d} | MSE: {mse:.4f} | Time: {running_time:.4f}s")
best_k_no_std = min(results, key=lambda x: x['MSE'])['K']
print(f"Best K (without standardization): {best_k_no_std}\n")


print("Question 2: KNN Regression with Standardization\n")
X_train_mean = X_train.mean()
X_train_std = X_train.std()
X_train_scaled = (X_train - X_train_mean) / X_train_std
X_test_scaled = (X_test - X_train_mean) / X_train_std

results_scaled = []
for k in range(1, 21):
    knn = KNNRegressor(k=k)
    
    start_time = time.time()
    knn.fit(X_train_scaled, y_train)
    y_pred_scaled = knn.predict(X_test_scaled)
    end_time = time.time()
    
    mse = mean_squared_error(y_test, y_pred_scaled)
    running_time = end_time - start_time
    
    results_scaled.append({
        'K': k,
        'MSE': mse,
        'Running Time (s)': running_time
    })
    
    print(f"K={k:2d} | MSE: {mse:.4f} | Time: {running_time:.4f}s")
best_k_with_std = min(results_scaled, key=lambda x: x['MSE'])['K']
print(f"Best K (with standardization): {best_k_with_std}\n")


print("Question 3: Comparison of Standardization Effect\n")
best_mse_no_std = min(results, key=lambda x: x['MSE'])['MSE']
best_mse_with_std = min(results_scaled, key=lambda x: x['MSE'])['MSE']
print(f"Best MSE without standardization: {best_mse_no_std:.4f}")
print(f"Best MSE with standardization: {best_mse_with_std:.4f}")
if best_mse_with_std < best_mse_no_std:
    print("\nYes, standardization improves performance.")
else:
    print("\nNo, standardization does not improve performance.")


In [ ]:

X_train, y_train = load_data(D_train)
model = LogisticRegression()
model.fit(X_train, y_train)


y_prob = model.predict_proba(X_train)[:, 1]
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_train, y_prob)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='ROC Curve')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier') 
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve for Logistic Regression on Training Data')
plt.legend()
plt.grid(True)
plt.show()


In [1]:
import os
os.chdir(r'D:\HKUST\5054_Statistical_Machine_Learning\Assignments\HW2')
import pandas as pd


train = pd.read_csv("adult_train.csv")
test = pd.read_csv("adult_test.csv")
train = train.dropna()
test = test.dropna()
print(test.columns.tolist())
print(train.columns.tolist())
print(train.head(10))

print(train['Class'].unique())
train['Class'] = train['Class'].map({'benign': 0, 'malignant': 1})
test['Class'] = test['Class'].map({'benign': 0, 'malignant': 1})

df = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Fare']].copy()
df = df.dropna()
print(df.columns.tolist())

Sex_dummies = pd.get_dummies(df['Sex'], prefix = 'Sex', dtype = int, drop_first = True)
Pclass_dummies = pd.get_dummies(df['Pclass'], prefix='Pclass', dtype=int, drop_first = True)

df = pd.concat([df, Sex_dummies, Pclass_dummies], axis=1)
original_cols = ['Age', 'SibSp', 'Fare']
all_feature_cols = original_cols + Sex_dummies.columns.tolist() + Pclass_dummies.columns.tolist()
print(all_feature_cols)
X = df[all_feature_cols]
y = df[['Survived']]
print(f"X shape:{X.shape}, y shape:{y.shape}")

X_train_full = train.drop(columns=['Id', 'Class'])
y_train = train['Class']
X_test_full = test.drop(columns=['Id', 'Class'])
y_test = test['Class']
selected_features = ['Cl.thickness', 'Cell.shape', 'Marg.adhesion', 'Bare.nuclei', 'Bl.cromatin']
X_train_5 = X_train_full[selected_features]
X_test_5 = X_test_full[selected_features]


print(f" Use all the predictors to fit a logistic regression")
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
lr_full = LogisticRegression(max_iter=1000)
lr_full.fit(X_train_full, y_train)
y_prob_full = lr_full.predict_proba(X_test_full)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_full)
lr_roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {lr_roc_auc:.4f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression (All Features)')
plt.legend()
plt.show()
print("LR_AUC (Full Features):", lr_roc_auc)


print(f" Use 5 predictors to fit logistic regression")
lr_five = LogisticRegression(max_iter=1000)
lr_five.fit(X_train_5, y_train)
y_prob_five = lr_five.predict_proba(X_test_5)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_five)
lr5_roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {lr5_roc_auc:.4f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression (Five Features)')
plt.legend()
plt.show()
print("LR_AUC (Five Features):", lr5_roc_auc)


from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
lda_full = LinearDiscriminantAnalysis()
lda_full.fit(X_train_full, y_train)
y_prob_lda_full = lda_full.predict_proba(X_test_full)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_lda_full)
lda_roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f'LDA (AUC = {lda_roc_auc:.4f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - LDA (Full Features)')
plt.legend()
plt.show()
print("LDA_AUC (Full Features):", lda_roc_auc)


lda_five = LinearDiscriminantAnalysis()
lda_five.fit(X_train_5, y_train)
y_prob_lda_five = lda_five.predict_proba(X_test_5)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_lda_five)
lda5_roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f'LDA (AUC = {lda5_roc_auc:.4f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - LDA (Five Features)')
plt.legend()
plt.show()
print("LDA_AUC (Five Features):", lda5_roc_auc)


qda_full = QuadraticDiscriminantAnalysis()
qda_full.fit(X_train_full, y_train)
y_prob_qda = qda_full.predict_proba(X_test_full)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_qda)
qda_roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f'QDA (AUC = {qda_roc_auc :.4f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - QDA (Full Features)')
plt.legend()
plt.show()
print("QDA_AUC (Full Features):", qda_roc_auc)


results = []
results.append({"Model": "logistic regression", "Features": "All", "AUC": lr_roc_auc})
results.append({"Model": "logistic regression", "Features": "Five", "AUC": lr5_roc_auc})
results.append({"Model": "LDA", "Features": "All", "AUC": lda_roc_auc})
results.append({"Model": "LDA", "Features": "Five", "AUC": lda5_roc_auc})
results.append({"Model": "QDA", "Features": "All", "AUC": qda_roc_auc})

df_auc = pd.DataFrame(results)
df_auc = df_auc.sort_values(by="AUC", ascending=False).reset_index(drop=True)
print(df_auc.round(6))


FileNotFoundError: [Errno 2] No such file or directory: 'adult_train.csv'

In [ ]:
import os
os.chdir(r'D:\HKUST\5054_Statistical_Machine_Learning\Assignments\HW2')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm

cars = pd.read_csv("cars.csv")
print(cars.columns.tolist())
print(cars.head())
X = cars[['speed']].values 
y = cars['dist'].values 

def poly_cv_mse(X, y, degrees, cv):
    mse_list = []
    for d in degrees:
        poly = PolynomialFeatures(degree=d, include_bias=False)
        X_poly = poly.fit_transform(X)
        model = LinearRegression()
        neg_mse_scores = cross_val_score(model, X_poly, y, cv=cv, scoring='neg_mean_squared_error')
        mse = -np.mean(neg_mse_scores) 
        mse_list.append(mse)
    return np.array(mse_list)

print(f"Use leave-one-out cross validation")
degrees = np.arange(1, 11) 
loo = LeaveOneOut()
loo_mse = poly_cv_mse(X, y, degrees, loo)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(degrees, loo_mse, 'bo-', linewidth=2, markersize=6)
plt.xlabel('Polynomial Degree')
plt.ylabel('LOO CV MSE')
plt.title('LOO CV Error for different Polynomial Degree')
plt.grid(True, alpha=0.3)
best_deg_loo = degrees[np.argmin(loo_mse)]
print(f"The best degree is {best_deg_loo}，MSE = {loo_mse.min():.2f}")


print(f"Use K-fold CV")
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
kf5_mse = poly_cv_mse(X, y, degrees, kf5)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(degrees, kf5_mse, 'ro-', linewidth=2, markersize=6)
plt.xlabel('Polynomial Degree')
plt.ylabel('5-Fold CV MSE')
plt.title('5-Fold CV Error for different Polynomial Degree')
plt.grid(True, alpha=0.3)
best_deg_kf5 = degrees[np.argmin(kf5_mse)]
print(f"The best degree is {best_deg_kf5}，MSE = {kf5_mse.min():.2f}")
plt.tight_layout()
plt.show()


def gaussian_kernel_predict(X_train, y_train, X_test, h):
    preds = []
    for x in X_test:
        distances = np.abs(X_train.flatten() - x) 
        weights = np.exp(- (distances ** 2) / (2 * h ** 2))  
        pred = np.sum(weights * y_train) / np.sum(weights)  
        preds.append(pred)
    return np.array(preds)
    
def knn_gaussian_cv_mse(X, y, h_values, cv):
    mse_list = []
    for h in h_values:
        fold_mse = []
        for train_idx, val_idx in cv.split(X): 
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            y_pred = gaussian_kernel_predict(X_tr, y_tr, X_val, h)
            fold_mse.append(mean_squared_error(y_val, y_pred))
        mse_list.append(np.mean(fold_mse))
    return np.array(mse_list)

h_values = np.linspace(0.5, 10, 50) 
print(f"from 0.5 to 10 totally 50 values")

loo_mse_knn = knn_gaussian_cv_mse(X, y, h_values, LeaveOneOut())
kf5_mse_knn = knn_gaussian_cv_mse(X, y, h_values, KFold(n_splits=5, shuffle=True, random_state=42))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(h_values, loo_mse_knn, 'bo-', linewidth=2, markersize=6)
plt.xlabel('Bandwidth h')
plt.ylabel('LOO CV MSE')
plt.title('KNN-Gaussian: LOO CV Error vs Bandwidth')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(h_values, kf5_mse_knn, 'ro-', linewidth=2, markersize=6)
plt.xlabel('Bandwidth h')
plt.ylabel('5-Fold CV MSE')
plt.title('KNN-Gaussian: 5-Fold CV Error vs Bandwidth')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


print(f"Compare the KNN-Gaussian kernel and polynomial regression")
best_h_loo = h_values[np.argmin(loo_mse_knn)]
best_h_kf5 = h_values[np.argmin(kf5_mse_knn)]
print(f"Use LOO CV, best bandwidth = {best_h_loo:.2f}，MSE = {loo_mse_knn.min():.2f}")
print(f"Use 5-Fold CV, best bandwidth = {best_h_kf5:.2f}，MSE = {kf5_mse_knn.min():.2f}")

best_deg_loo = degrees[np.argmin(loo_mse)]
best_deg_kf5 = degrees[np.argmin(kf5_mse)]
best_h_loo = h_values[np.argmin(loo_mse_knn)]
best_h_kf5 = h_values[np.argmin(kf5_mse_knn)]

print(f"\nUnder LOOCV")
print(f"Best polynomial regression (deg={best_deg_loo}): MSE = {loo_mse.min():.2f}")
print(f"Best KNN-Gaussian kernel (h={best_h_loo:.2f}): MSE = {loo_mse_knn.min():.2f}")
if loo_mse.min() < loo_mse_knn.min():
    print("Under the LOOCV, polynomial regression have better performance.")
    better_model_loo = "Polynomial"
    best_mse_loo = loo_mse.min()
else:
    print("Under the LOOCV, KNN-Gaussian kernel have better performance.")
    better_model_loo = "KNN-Gaussian"
    best_mse_loo = loo_mse_knn.min()

print(f"\nUnder 5-Fold CV")
print(f"Best polynomial regression (deg={best_deg_kf5}): MSE = {kf5_mse.min():.2f}")
print(f"Best KNN-Gaussian kernel (h={best_h_kf5:.2f}): MSE = {kf5_mse_knn.min():.2f}")
if kf5_mse.min() < kf5_mse_knn.min():
    print("Under the 5-Fold CV, polynomial regression have better performance.")
    better_model_kf5 = "Polynomial"
    best_mse_kf5 = kf5_mse.min()
else:
    print("Under the 5-Fold CV, KNN-Gaussian kernel have better performance.")
    better_model_kf5 = "KNN-Gaussian"
    best_mse_kf5 = kf5_mse_knn.min()

print(f"\nFinal result")
if better_model_loo == better_model_kf5:
    print(f"Both CV methods (LOO and 5-Fold) indicate that the {better_model_loo} model performs better (has a smaller MSE).")
else:
    print(f"LOO CV suggests the {better_model_loo} model is better, whereas 5-Fold CV suggests the {better_model_kf5} model is better.")


In [ ]:
import os
os.chdir(r'D:\HKUST\5054_Statistical_Machine_Learning\Assignments\HW2')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.utils import resample
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm

titanic = pd.read_csv("titanic.csv")
print(titanic.columns.tolist())
print(titanic.shape)

df = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Fare']].copy()
df = df.dropna()
print(df.columns.tolist())

Sex_dummies = pd.get_dummies(df['Sex'], prefix = 'Sex', dtype = int, drop_first = True)
Pclass_dummies = pd.get_dummies(df['Pclass'], prefix='Pclass', dtype=int, drop_first = True)

df = pd.concat([df, Sex_dummies, Pclass_dummies], axis=1)
original_cols = ['Age', 'SibSp', 'Fare']
all_feature_cols = original_cols + Sex_dummies.columns.tolist() + Pclass_dummies.columns.tolist()
print(all_feature_cols)
X = df[all_feature_cols]
y = df[['Survived']]
print(f"X shape:{X.shape}, y shape:{y.shape}")


print(f"LogisticRegression")
X_sm = sm.add_constant(X)
model = sm.Logit(y, X_sm)
result = model.fit()
print(result.summary())

coef_sex = result.params['Sex_male']
ci_sex = result.conf_int().loc['Sex_male']
coef_pclass3 = result.params['Pclass_3']
ci_pclass3 = result.conf_int().loc['Pclass_3']
print(f"Sex (male) coefficient: {coef_sex:.4f}, 95% CI: [{ci_sex[0]:.4f}, {ci_sex[1]:.4f}]")
print(f"Pclass (3rd) coefficient: {coef_pclass3:.4f}, 95% CI: [{ci_pclass3[0]:.4f}, {ci_pclass3[1]:.4f}]")


print(f"Bootstrap with 1000 repetitions")
def bootstrap_ci(model, X, y, n_iterations=1000, alpha=0.05):
    coefs = []     
    for _ in range(n_iterations):
        sample = df.sample(frac=1, replace=True)
        X_sample = sample[['Age', 'SibSp', 'Fare', 'Sex_male', 'Pclass_2', 'Pclass_3']]
        X_sample = sm.add_constant(X_sample)
        y_sample = sample['Survived']
        
        model_sample = sm.Logit(y_sample, X_sample)
        result_sample = model_sample.fit(disp=0)
        coefs.append(result_sample.params)
    
    coefs_df = pd.DataFrame(coefs)
    lower = coefs_df.quantile(alpha/2)
    upper = coefs_df.quantile(1-alpha/2)
    return lower, upper

lower, upper = bootstrap_ci(model, X, y)
print("Bootstrap 95% Confidence Intervals:")
print("Sex (male): [{}, {}]".format(lower['Sex_male'], upper['Sex_male']))
print("Pclass (3rd): [{}, {}]".format(lower['Pclass_3'], upper['Pclass_3']))


survival_by_sex = df.groupby('Sex')['Survived'].mean()
survival_by_pclass = df.groupby('Pclass')['Survived'].mean()
survival_by_age_group = df.groupby(df['Age'] // 10 * 10)['Survived'].mean() 
print("\nThe survival rate divided by sex:")
print(survival_by_sex)
print("\nThe survival rate divided by Pclass:")
print(survival_by_pclass)
print("\nThe survival rate divided by age (10-year bins)")
print(survival_by_age_group)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].bar(['Female', 'Male'], survival_by_sex.values, color=['pink', 'blue'])
ax[0].set_title('Survival Rate by Sex')
ax[0].set_ylabel('Survival Rate')
ax[1].bar(['1st', '2nd', '3rd'], survival_by_pclass.values, color=['gold', 'silver', 'brown'])
ax[1].set_title('Survival Rate by Pclass')
ax[1].set_ylabel('Survival Rate')
ax[2].bar(survival_by_age_group.index.astype(str), survival_by_age_group.values, color='lightcoral')
ax[2].set_title('Survival Rate by Age Group (10-year bins)')
ax[2].set_ylabel('Survival Rate')
ax[2].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


print(f"prediction interval")
X_train = df[all_feature_cols].iloc[1:].values
X_train = sm.add_constant(X_train)  
y_train = df[['Survived']].iloc[1:].values
X_test = df[all_feature_cols].iloc[0].values.reshape(1, -1)
X_test = sm.add_constant(X_test, has_constant = 'add')

print(f"X_test shape after adding constant: {X_test.shape}")  
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}",) 

def bootstrap_prediction_interval_logit(X_train, y_train, X_test, n_iterations=1000, alpha=0.05):
    predictions = []
    
    for _ in range(n_iterations):
        X_sample, y_sample = resample(X_train, y_train, replace=True) 
        
        model_sample = sm.Logit(y_sample, sm.add_constant(X_sample))  
        result_sample = model_sample.fit(disp=0)

        predictions.append(result_sample.predict(sm.add_constant(X_test))[0])  
    lower = np.percentile(predictions, alpha/2*100)
    upper = np.percentile(predictions, (1-alpha/2)*100)
    return lower, upper

lower, upper = bootstrap_prediction_interval_logit(X_train, y_train, X_test)
print(f"Logistic Regression - Bootstrap 95% Prediction Interval for Test Point: [{lower:.4f}, {upper:.4f}]")


print(f"QDA with prediction interval")
X_train = df[all_feature_cols].iloc[1:].values
y_train = df[['Survived']].iloc[1:].values

X_test = df[all_feature_cols].iloc[0].values.reshape(1, -1)
print(f"X_test shape after adding constant: {X_test.shape}") 


def bootstrap_prediction_interval_qda(X_train, y_train, X_test, n_iterations=1000, alpha=0.05):
    predictions = []
    
    for _ in range(n_iterations):
        X_sample, y_sample = resample(X_train, y_train, replace=True) 
        
        qda_sample = QuadraticDiscriminantAnalysis()
        qda_sample.fit(X_sample, y_sample.ravel()) 
        
        predictions.append(qda_sample.predict_proba(X_test)[0, 1])

    lower = np.percentile(predictions, alpha/2*100)
    upper = np.percentile(predictions, (1-alpha/2)*100)
    return lower, upper

lower_qda, upper_qda = bootstrap_prediction_interval_qda(X_train, y_train, X_test)
print(f"QDA - Bootstrap 95% Prediction Interval for Test Point: [{lower_qda:.4f}, {upper_qda:.4f}]")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load the dataset
adv = pd.read_csv('Datasets/Advertising.csv')
print(adv.head(10))

# Scatter plot to check the linearity
plt.figure(figsize=(8, 6))
plt.scatter(adv['TV'], adv['Sales'], c='black', s=40)
plt.xlabel('TV')
plt.ylabel('Sales')
plt.title('Sales vs. TV')
plt.show()

# Linear regression model 1: Sales ~ TV
X1 = adv[['TV']]
y = adv['Sales']
X1 = sm.add_constant(X1)  # Adds a constant term to the predictor
model1 = sm.OLS(y, X1).fit()
print(model1.summary())

# Plotting the regression line
plt.figure(figsize=(8, 6))
plt.scatter(adv['TV'], adv['Sales'], c='black', s=40)
plt.plot(adv['TV'], model1.predict(X1), color='darkorange', linewidth=2)
plt.xlabel('TV')
plt.ylabel('Sales')
plt.title('Sales vs. TV with Regression Line')
plt.show()

# Linear regression model 2: Sales ~ all predictors
X2 = adv.drop(columns='Sales')
X2 = sm.add_constant(X2)  # Adds a constant term to the predictor
model2 = sm.OLS(y, X2).fit()
print(model2.summary())


# ANOVA
mod1 = smf.ols('Sales ~ TV', data=adv).fit()
mod2 = smf.ols('Sales ~ TV + Radio + Newspaper', data=adv).fit()
anova_results = sm.stats.anova_lm(mod1, mod2)
print(anova_results)

# Confidence interval
conf_interval = mod1.conf_int(alpha=0.05)
print(conf_interval)

# Prediction interval
new_obs = pd.DataFrame({'TV': [145.3], 'Radio': [51.1], 'Newspaper': [33.9]})
pred_interval = mod1.get_prediction(new_obs[['TV']]).summary_frame(alpha=0.05)
print(pred_interval)

In [ ]:
######## KNN regression

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor

# Generate training data
np.random.seed(0)
train_x = pd.DataFrame({'x': np.random.uniform(-1, 1, 40)})
train_y = 2 * train_x['x'] + 2 + np.random.normal(0, 0.3, 40)

# KNN regression model
knnmod1 = KNeighborsRegressor(n_neighbors=3)
knnmod1.fit(train_x, train_y)

# Generate test data
test_x = pd.DataFrame({'x': np.arange(-1, 1, 0.001)})
test_y = knnmod1.predict(test_x)

# Plot
plt.scatter(train_x['x'], train_y, c='black', s=40, label='Training data')
plt.plot(test_x['x'], 2 * test_x['x'] + 2, color='orange', linewidth=4, label='True function')
plt.plot(test_x['x'], test_y, color='blue', linewidth=4, label='KNN prediction')
plt.title('KNN with k=3 to fit y=2x+2+eps')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
############################### simulation showing that (hatbeta-beta)/sd(beta) indeed follows t distribution.

import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

# Parameters
n_simulations = 500
n_samples = 100
true_beta = np.array([1.5, -2.0, 3.0])
t_statistics = []

# Simulation loop
np.random.seed(0)
for _ in range(n_simulations):
    X = np.random.normal(size=(n_samples, 3))
    y = X @ true_beta + np.random.normal(scale=1.0, size=n_samples)
    
    # Fit the model
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    
    # Calculate t-statistics
    estimated_beta = model.params[1:]  # Exclude intercept
    se_beta = model.bse[1:]  # Standard errors
    t_stat = (estimated_beta - true_beta) / se_beta
    t_statistics.extend(t_stat)

# Compare with t-distribution
df = model.df_resid
x = np.linspace(-4, 4, 100)
t_dist = stats.t(df)

plt.hist(t_statistics, bins=30, density=True, alpha=0.6, color='g', label='t-statistics')
plt.plot(x, t_dist.pdf(x), 'r-', label=f't-distribution (df={int(df)})')
plt.xlabel('t-statistic')
plt.ylabel('Density')
plt.title('T-statistics vs. T-distribution')
plt.legend()
plt.show()

In [ ]:
################### Multiple linear regression, only X1 and X2 are useful, report the AIC, BIC, Mallow's Cp and Adjusted Rsquare

import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations

# Generate synthetic data
np.random.seed(19)
n_samples = 100
X = np.random.normal(size=(n_samples, 4))
true_beta = np.array([2.0, -1.5, 0, 0])  # Only first 2 predictors are useful
y = X @ true_beta + np.random.normal(scale=1.0, size=n_samples)

# DataFrame for predictors
df = pd.DataFrame(X, columns=[f'X{i+1}' for i in range(X.shape[1])])

# Function to calculate Mallow's Cp
def mallows_cp(model, sigma_squared):
    p = model.df_model + 1
    rss = model.ssr
    return (rss / sigma_squared) - (len(y) - 2 * p)

# Fit models of different sizes
results = []

for k in range(1, 5):
    for combo in combinations(df.columns, k):
        X_subset = sm.add_constant(df[list(combo)])
        model = sm.OLS(y, X_subset).fit()
        
        # Calculate statistics
        aic = model.aic
        bic = model.bic
        adj_r_squared = model.rsquared_adj
        sigma_squared = model.mse_resid
        cp = mallows_cp(model, sigma_squared)
        
        results.append({
            'Predictors': combo,
            'AIC': aic,
            'BIC': bic,
            "Mallow's Cp": cp,
            'Adjusted R^2': adj_r_squared
        })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df.sort_values('AIC').reset_index(drop=True))

In [ ]:
################################## polynomial regression on auto data using validation approach

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_openml

# Load dataset
Auto = pd.read_csv("Auto.csv")  # Replace with the actual path to the dataset

# Display the first 10 rows
print(Auto.head(10))

# Validation approach
n = Auto.shape[0]
np.random.seed(1997)
trainid = np.random.choice(n, n // 2, replace=False)

# Convert to numpy arrays
X = Auto[['horsepower']].values
y = Auto['mpg'].values

# Linear regression
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, random_state=1997)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean squared error (linear): {mse}")

# Polynomial regression (degree 2)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

X_train_poly, X_test_poly, y_train, y_test = train_test_split(X_poly, y, train_size=0.5, random_state=1997)

lin_reg_poly = LinearRegression()
lin_reg_poly.fit(X_train_poly, y_train)
y_pred_poly = lin_reg_poly.predict(X_test_poly)
mse_poly = mean_squared_error(y_test, y_pred_poly)
print(f"Mean squared error (quadratic): {mse_poly}")

# Polynomial regression (degree 3)
poly3 = PolynomialFeatures(degree=3)
X_poly3 = poly3.fit_transform(X)

X_train_poly3, X_test_poly3, y_train, y_test = train_test_split(X_poly3, y, train_size=0.5, random_state=1997)

lin_reg_poly3 = LinearRegression()
lin_reg_poly3.fit(X_train_poly3, y_train)
y_pred_poly3 = lin_reg_poly3.predict(X_test_poly3)
mse_poly3 = mean_squared_error(y_test, y_pred_poly3)
print(f"Mean squared error (cubic): {mse_poly3}")

################# plot

# Sort the data
sorted_indices = np.argsort(X, axis=0).flatten()
X_sorted = X[sorted_indices]
y_sorted = y[sorted_indices]

# Function to fit and plot polynomial regression
def plot_polynomial_regression(X, y, degree):
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X)
    lin_reg = LinearRegression()
    lin_reg.fit(X_poly, y)
    y_pred = lin_reg.predict(X_poly)
    return y_pred

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(X_sorted, y_sorted, color='blue', label='Original data', alpha=0.5)

# Linear regression
y_pred_linear = plot_polynomial_regression(X_sorted, y_sorted, degree=1)
plt.plot(X_sorted, y_pred_linear, color='red', label='Linear fit', linewidth=2)

# Quadratic regression
y_pred_quadratic = plot_polynomial_regression(X_sorted, y_sorted, degree=2)
plt.plot(X_sorted, y_pred_quadratic, color='green', label='Quadratic fit', linewidth=2)

# Cubic regression
y_pred_cubic = plot_polynomial_regression(X_sorted, y_sorted, degree=3)
plt.plot(X_sorted, y_pred_cubic, color='purple', label='Cubic fit', linewidth=2)

# Labels and legend
plt.xlabel('Horsepower')
plt.ylabel('MPG')
plt.title('Polynomial Regression Fits')
plt.legend()
plt.show()


In [ ]:
########## LOOC on auto data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.pipeline import make_pipeline

# Load dataset
Auto = pd.read_csv("Auto.csv")  # Replace with the actual path to the dataset

######Linear model and LOOC
# Define features and target variable
X = Auto[['horsepower']].values  # Predictor
y = Auto['mpg'].values           # Response
linear_model = LinearRegression()
linear_model.fit(X, y)
print("Linear Regression Coefficients:")
print(f"Intercept: {linear_model.intercept_}")
print(f"Slope (horsepower): {linear_model.coef_[0]}")

loo = LeaveOneOut()
cv_errors = cross_val_score(linear_model, X, y, cv=loo, scoring='neg_mean_squared_error')
mse = -cv_errors
loocv_error = mse.mean()
print(f"LOOCV Estimate of Prediction Error (MSE): {loocv_error:.4f}")

############## Polynomial regression and LOOC
degrees = range(1, 11)
cv_errors_poly = []
for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    scores = cross_val_score(model, X, y, cv=loo, scoring='neg_mean_squared_error')
    avg_mse = -scores.mean()
    cv_errors_poly.append(avg_mse)
    print(f"Degree {degree}: LOOCV MSE = {avg_mse:.4f}")

################# Visualization
plt.figure(figsize=(10, 6))
plt.plot(degrees, cv_errors_poly, marker='o', linestyle='-', color='b')
plt.title('LOOCV Error vs. Degree of Polynomial', fontsize=16)
plt.xlabel('Degree of Polynomial', fontsize=14)
plt.ylabel('LOOCV Mean Squared Error', fontsize=14)
plt.xticks(degrees)
plt.grid(True)
plt.show()

In [ ]:
########## best subset selection for linear model on Credit data
import pprint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# 1. Load the Credit Dataset

Credit = pd.read_csv('Credit.csv')
Credit = Credit.drop('ID', axis=1)
print(Credit.head(10))
print(Credit.shape)
[num_samples,num_predictors]=Credit.shape

Credit['Gender'] = Credit['Gender'].map({' Male': 1, 'Female': 0})
Credit['Student'] = Credit['Student'].map({'Yes': 1, 'No': 0})
Credit['Married'] = Credit['Married'].map({'Yes': 1, 'No': 0})
Credit['Ethnicity'] = Credit['Ethnicity'].map({"Asian": 2, 'Caucasian': 1, 'African American': 0})

# 2. Scatterplot Matrix
selected_columns=['Balance','Education','Age','Cards','Rating','Limit','Income']
# Plot scatterplot matrix
sns.set(style="ticks", color_codes=True)
plt.figure(figsize=(12, 10))
sns.pairplot(Credit[selected_columns], diag_kind='kde')
plt.suptitle("Scatterplot Matrix", y=1.02)
plt.show()

# 3. Best Subset Selection
# --------------------------------
# We'll perform exhaustive best subset selection for models with up to 10 predictors.

# Define target variable and predictors
target = 'Balance'
predictors = [col for col in Credit.columns if col != target]
print(predictors)
# Set maximum number of predictors
nvmax = 10

# Initialize lists to store model statistics
model_stats = []

# Iterate over number of predictors from 1 to nvmax
for k in range(nvmax ,0,-1):
    print(f"Evaluating models with {k} predictors...")
    # Generate all possible combinations of predictors of size k
    subsets = list(combinations(predictors, k))
    for subset in subsets:
        # Prepare the design matrix
        X_subset = Credit[list(subset)]
        X_subset = sm.add_constant(X_subset)  # Adds a constant term to the predictor
        # Fit the model using Ordinary Least Squares
        model = sm.OLS(Credit[target], X_subset).fit()
        
        # Calculate RSS
        rss = sum(model.resid ** 2)
        
        # Calculate R-squared
        rsq = model.rsquared
        
        # Calculate Adjusted R-squared
        adj_rsq = model.rsquared_adj
        
        # Calculate BIC
        bic = model.bic
        
        # Calculate Mallow's Cp
        # Cp = RSS / MSE_full - (n - 2p)
        # Estimate MSE from the full model (using all predictors)
        # This requires the full model's RSS and degrees of freedom
        # Fit the full model if k == nvmax
        if k == nvmax:
            full_model = model
            mse_full = full_model.mse_resid
        # For other models, use mse_full from the full model
        cp = (rss / mse_full) - (num_samples - 2 * k)
        
        # Store the statistics
        model_stats.append({
            'Num_Predictors': k,
            'Predictors': subset,
            'RSS': rss,
            'R_squared': rsq,
            'Adj_R_squared': adj_rsq,
            'BIC': bic,
            'Cp': cp
        })

# Convert the list of dictionaries to a DataFrame
model_stats_df = pd.DataFrame(model_stats)


# 4. Identifying Best Models per Number of Predictors
# --------------------------------
# For each k, find the model with the lowest RSS, highest R-squared, etc.

best_models = model_stats_df.loc[model_stats_df.groupby('Num_Predictors')['RSS'].idxmin()]

# 5. Plotting Model Statistics
# --------------------------------

# Function to plot metrics
def plot_metric(metric, ylabel, title):
    plt.figure(figsize=(10, 6))
    
    # Plot all models in grey
    sns.scatterplot(data=model_stats_df, x='Num_Predictors', y=metric, color='grey', alpha=0.3, label='All Models')
    
    # Plot best models in red
    sns.lineplot(data=best_models, x='Num_Predictors', y=metric, color='red', marker='o', label='Best Model')
    
    plt.title(title)
    plt.xlabel('Number of Predictors')
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Plot RSS
plot_metric(metric='RSS', ylabel='Residual Sum of Squares (RSS)', title='Best Subset Selection: RSS by Number of Predictors')

# Plot R-squared
plot_metric(metric='R_squared', ylabel='R-squared', title='Best Subset Selection: R-squared by Number of Predictors')

# Plot Adjusted R-squared
plot_metric(metric='Adj_R_squared', ylabel='Adjusted R-squared', title='Best Subset Selection: Adjusted R-squared by Number of Predictors')

# Plot BIC
plot_metric(metric='BIC', ylabel='Bayesian Information Criterion (BIC)', title='Best Subset Selection: BIC by Number of Predictors')

# Plot Mallow's Cp
plot_metric(metric='Cp', ylabel="Mallow's Cp", title="Best Subset Selection: Mallow's Cp by Number of Predictors")

# 6. Display Best Models with Highest Metrics
# --------------------------------
# Optionally, display the best models' details
print("\nBest Models for Each Number of Predictors:")
print(best_models[['Num_Predictors', 'Predictors', 'RSS', 'R_squared', 'Adj_R_squared', 'BIC', 'Cp']])

# 7. Interpretation
# --------------------------------
print("\nInterpretation of Plots:")
print("""
1. **RSS Plot:** Shows the Residual Sum of Squares for all models. The best model for each number of predictors has the lowest RSS.

2. **R-squared Plot:** Indicates how well each model explains the variability of the target variable. The best model maximizes R-squared.

3. **Adjusted R-squared Plot:** Adjusts R-squared for the number of predictors, preventing overestimation when adding more variables. The best model has the highest Adjusted R-squared.

4. **BIC Plot:** Penalizes models with more predictors to prevent overfitting. The best model typically has the lowest BIC.

5. **Mallow's Cp Plot:** Helps in identifying models with significant predictors. A model with Cp ≈ p (number of predictors) is considered ideal. The best model aims for Cp close to the number of predictors.
""")

In [ ]:
######## backward subset selection on Credit data
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Function to compute evaluation metrics
def compute_metrics(y, y_pred, p, n, sigma2):
    rss = np.sum((y - y_pred) ** 2)
    tss = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - rss / tss
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    bic = n * np.log(rss / n) + p * np.log(n)
    cp = rss / sigma2 - n + 2 * p
    return rss, adj_r2, bic, cp

# ---------------------------------------------------
# 1. Load and Preview the Credit Dataset
# ---------------------------------------------------
Credit = pd.read_csv('Credit.csv')
Credit = Credit.drop('ID', axis=1)
print(Credit.head(10))
[num_samples,num_predictors]=Credit.shape

Credit['Gender'] = Credit['Gender'].map({' Male': 1, 'Female': 0})
Credit['Student'] = Credit['Student'].map({'Yes': 1, 'No': 0})
Credit['Married'] = Credit['Married'].map({'Yes': 1, 'No': 0})
Credit['Ethnicity'] = Credit['Ethnicity'].map({"Asian": 2, 'Caucasian': 1, 'African American': 0})
# ---------------------------------------------------
# 2. Backward Subset Selection
# ---------------------------------------------------

# Define response and predictors
y = Credit['Balance']
X = Credit.drop('Balance', axis=1)

# Fit the full model to estimate sigma^2 for Mallows' Cp
X_full = sm.add_constant(X)
full_model = sm.OLS(y, X_full).fit()
sigma2 = full_model.mse_resid

# Initialize lists to store metrics and selected features
min_features = 1
rss_list = []
adjr2_list = []
bic_list = []
cp_list = []
selected_features = []

# Initialize the list of selected and remaining features
current_features = list(X.columns)  # Start with all features for backward selection
remaining_features = list(X.columns)

n = len(y)  # Number of observations

# Perform Backward Subset Selection
for k in range(len(current_features), min_features , -1):
    best_rss = np.inf
    worst_feature = None
    best_model = None
    
    if k == 0:
        break  # No features left to remove
    
    # Iterate over current features to find the worst one to remove
    for feature in current_features:
        features_to_try = current_features.copy()
        features_to_try.remove(feature)
        X_subset = X[features_to_try]
        X_subset_const = sm.add_constant(X_subset)
        model = sm.OLS(y, X_subset_const).fit()
        rss = model.ssr  # Residual Sum of Squares
        
        if rss < best_rss:
            best_rss = rss
            worst_feature = feature
            best_model = model
    
    # Update the list of selected features by removing the worst feature
    current_features.remove(worst_feature)
    
    # Compute metrics for the best model at this step
    p = len(current_features)  # Number of predictors
    y_pred = best_model.fittedvalues
    rss, adjr2, bic, cp = compute_metrics(y, y_pred, p, n, sigma2)
    
    # Store the metrics and selected features
    rss_list.append(rss)
    adjr2_list.append(adjr2)
    bic_list.append(bic)
    cp_list.append(cp)
    selected_features.append(tuple(current_features))
    
    print(f"Number of variables: {p}, Removed feature: {worst_feature}, Remaining features: {current_features}")

# Reverse the lists to align with the number of variables (from 1 to max_features)
rss_list = rss_list[::-1]
adjr2_list = adjr2_list[::-1]
bic_list = bic_list[::-1]
cp_list = cp_list[::-1]
selected_features = selected_features[::-1]

# ---------------------------------------------------
# 3. Visualization of Selection Metrics
# ---------------------------------------------------

plt.figure(figsize=(14, 12))

# Plot RSS
plt.subplot(2, 2, 1)
plt.plot(range(1, len(rss_list) +1), rss_list, marker='o', color='red', linewidth=2)
plt.xlabel('Number of Variables')
plt.ylabel('Residual Sum of Squares (RSS)')
plt.title('Backward Subset Selection - RSS')
plt.grid(True)

# Plot Adjusted R-squared
plt.subplot(2, 2, 2)
plt.plot(range(1, len(adjr2_list) +1), adjr2_list, marker='o', color='green', linewidth=2)
plt.xlabel('Number of Variables')
plt.ylabel('Adjusted R-Squared')
plt.title('Backward Subset Selection - Adjusted R²')
plt.grid(True)

# Plot BIC
plt.subplot(2, 2, 3)
plt.plot(range(1, len(bic_list) +1), bic_list, marker='o', color='blue', linewidth=2)
plt.xlabel('Number of Variables')
plt.ylabel('Bayesian Information Criterion (BIC)')
plt.title('Backward Subset Selection - BIC')
plt.grid(True)

# Plot Mallows' Cp
plt.subplot(2, 2, 4)
plt.plot(range(1, len(cp_list) +1), cp_list, marker='o', color='purple', linewidth=2)
plt.xlabel('Number of Variables')
plt.ylabel("Mallows' Cp")
plt.title("Backward Subset Selection - Mallows' Cp")
plt.grid(True)

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# 4. Display Selected Features at Each Step
# ---------------------------------------------------

print("\nSelected features at each step (from 1 to max number of variables):")
for i, features in enumerate(selected_features, 1):
    print(f"Variables: {i}, Features: {features}")

In [ ]:
##### LASSO on Credit data
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ---------------------------------------------------
# 1. Load and Preview the Credit Dataset
# ---------------------------------------------------

Credit = pd.read_csv('Credit.csv')
Credit = Credit.drop('ID', axis=1)
print(Credit.head(10))
[num_samples,num_predictors]=Credit.shape

Credit['Gender'] = Credit['Gender'].map({' Male': 1, 'Female': 0})
Credit['Student'] = Credit['Student'].map({'Yes': 1, 'No': 0})
Credit['Married'] = Credit['Married'].map({'Yes': 1, 'No': 0})
Credit['Ethnicity'] = Credit['Ethnicity'].map({"Asian": 2, 'Caucasian': 1, 'African American': 0})

n_features = 10  # Number of predictors
feature_names = Credit.columns[:-1].tolist()

# ---------------------------------------------------
# 2. LASSO Regression
# ---------------------------------------------------

# Define response and predictors
y = Credit['Balance'].values
X_features = Credit.drop('Balance', axis=1).values
n_samples=len(y)

# Standardize the predictors
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Define the lambda grid (equivalent to alpha in scikit-learn's Lasso)
lambda_grid = 10 ** np.linspace(4, -2, 100)

# Initialize an array to store coefficients for each lambda
coef_matrix = np.zeros((len(lambda_grid), X_features.shape[1]))

# Fit LASSO models for each lambda and store the coefficients
for i, lam in enumerate(lambda_grid):
    lasso = Lasso(alpha=lam, fit_intercept=True, max_iter=10000)
    lasso.fit(X_scaled, y)
    coef_matrix[i, :] = lasso.coef_

# Extract coefficients for the 50th and 60th models
# Note: Python uses 0-based indexing
index_50 = 49
index_60 = 59

coef_50 = coef_matrix[index_50, :]
coef_60 = coef_matrix[index_60, :]

print(f"\nLambda at 50th index: {lambda_grid[index_50]:.4f}")
print(f"Coefficients at 50th lambda:")
print(dict(zip(feature_names, coef_50)))
print(f"L1 norm of coefficients at 50th lambda: {np.linalg.norm(coef_50, 1):.4f}")

print(f"\nLambda at 60th index: {lambda_grid[index_60]:.4f}")
print(f"Coefficients at 60th lambda:")
print(dict(zip(feature_names, coef_60)))
print(f"L1 norm of coefficients at 60th lambda: {np.linalg.norm(coef_60, 1):.4f}")

# ---------------------------------------------------
# 3. Visualization of Coefficient Norms and Solution Path
# ---------------------------------------------------

# Calculate L1 norms of coefficients for all lambda values
l1_norms = np.linalg.norm(coef_matrix, ord=1, axis=1)

# Plot L1 norm of coefficients vs lambda index
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(lambda_grid) + 1), l1_norms, marker='o', color='green', linewidth=2)
plt.xlabel('Lambda Index')
plt.ylabel('L1 Norm of Coefficients')
plt.title('LASSO Regression: L1 Norm of Coefficients vs Lambda Index')
plt.grid(True)
plt.show()

# Plot solution path for selected features
# Adjust the feature indices based on your dataset. Here, we'll select features Feature3, Feature4, Feature5, Feature10
# which correspond to indices 2, 3, 4, 9 (0-based indexing)
selected_features = feature_names
selected_indices = [feature_names.index(feat) for feat in selected_features]

plt.figure(figsize=(10, 6))
for idx, feature in zip(selected_indices, selected_features):
    plt.plot(range(1, len(lambda_grid) + 1), coef_matrix[:, idx], label=feature, linewidth=2)

plt.xlabel('Lambda Index')
plt.ylabel('Coefficient Value')
plt.title('LASSO Regression: Solution Path')
plt.legend()
plt.grid(True)
plt.show()

# ---------------------------------------------------
# 4. Prediction and Evaluation
# ---------------------------------------------------

# Split the data into training and testing sets
# Using 50% of the data for training as in the R code
np.random.seed(1)
train_indices = np.random.choice(range(n_samples), size=n_samples // 2, replace=False)
test_indices = np.array(list(set(range(n_samples)) - set(train_indices)))

X_train, X_test = X_scaled[train_indices, :], X_scaled[test_indices, :]
y_train, y_test = y[train_indices], y[test_indices]

# Fit LASSO on training data with the predefined lambda grid
lasso_models = []
for lam in lambda_grid:
    lasso = Lasso(alpha=lam, fit_intercept=True, max_iter=10000)
    lasso.fit(X_train, y_train)
    lasso_models.append(lasso)

# Predict on test data with lambda=50 (find the corresponding lambda in the grid)
# In R's glmnet, lambda is indexed starting at 1, but in Python it's 0-based
# Assuming the user wants to use the original index, lambda=50 -> index=49
# Here, lambda=50 is not in grid as grid has 100 values from 10^4 to 10^-2
# Instead, to demonstrate, use the 50th lambda
lambda_50 = lambda_grid[index_50]
lasso_pred_50 = lasso_models[index_50].predict(X_test)
mse_lambda_50 = mean_squared_error(y_test, lasso_pred_50)
print(f"\n10-fold CV Error with lambda={lambda_50:.4f}: {mse_lambda_50:.4f}")

# Predict on test data with lambda=20 (find the closest lambda in the grid)
lambda_20 = 20
closest_lambda_20 = min(lambda_grid, key=lambda x: abs(x - lambda_20))
index_lambda_20 = np.where(lambda_grid == closest_lambda_20)[0][0]
lasso_pred_20 = lasso_models[index_lambda_20].predict(X_test)
mse_lambda_20 = mean_squared_error(y_test, lasso_pred_20)
print(f"10-fold CV Error with lambda=20 (using closest lambda={closest_lambda_20:.4f}): {mse_lambda_20:.4f}")

# ---------------------------------------------------
# 5. Cross-Validation to Choose the Best Lambda
# ---------------------------------------------------

# Initialize LassoCV for cross-validation (10-fold CV by default)
lasso_cv = LassoCV(alphas=lambda_grid, cv=10, max_iter=10000, random_state=1)
lasso_cv.fit(X_train, y_train)

# Best lambda that minimizes CV error
best_lambda = lasso_cv.alpha_
print(f"\nBest lambda from cross-validation: {best_lambda:.4f}")

# Predict on test data using the best lambda
lasso_best = Lasso(alpha=best_lambda, fit_intercept=True, max_iter=10000)
lasso_best.fit(X_train, y_train)
lasso_pred_best = lasso_best.predict(X_test)
mse_best_lambda = mean_squared_error(y_test, lasso_pred_best)
print(f"10-fold CV Error with best lambda={best_lambda:.4f}: {mse_best_lambda:.4f}")

# ---------------------------------------------------
# 6. Fit LASSO on the Full Dataset with Best Lambda
# ---------------------------------------------------

lasso_full = Lasso(alpha=best_lambda, fit_intercept=True, max_iter=10000)
lasso_full.fit(X_scaled, y)
full_coefficients = lasso_full.coef_

print(f"\nLASSO Regression Coefficients on Full Dataset with lambda={best_lambda:.4f}:")
print(dict(zip(feature_names, full_coefficients)))

# Optionally, include the intercept
print(f"Intercept: {lasso_full.intercept_:.4f}")

# Display non-zero coefficients
non_zero_coefficients = {feat: coef for feat, coef in zip(feature_names, full_coefficients) if coef != 0}
print("\nNon-zero coefficients:")
print(non_zero_coefficients)

# ---------------------------------------------------
# 7. Fit LASSO with a Larger Lambda (e.g., lambda=20)
# ---------------------------------------------------

# As before, find the closest lambda in the grid to 20
lasso_lambda_20 = Lasso(alpha=closest_lambda_20, fit_intercept=True, max_iter=10000)
lasso_lambda_20.fit(X_scaled, y)
coeff_lambda_20 = lasso_lambda_20.coef_

print(f"\nLASSO Coefficients with lambda={closest_lambda_20:.4f}:")
print(dict(zip(feature_names, coeff_lambda_20)))

# Display non-zero coefficients
non_zero_coeff_lambda_20 = {feat: coef for feat, coef in zip(feature_names, coeff_lambda_20) if coef != 0}
print("\nNon-zero coefficients with lambda=20:")
print(non_zero_coeff_lambda_20)

In [ ]:
####### Ridge regression on Credit Data
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ---------------------------------------------------
# 1. Load and Preview the Credit Dataset
# ---------------------------------------------------

Credit = pd.read_csv('Credit.csv')
Credit = Credit.drop('ID', axis=1)
print(Credit.head(10))
[num_samples,num_predictors]=Credit.shape

Credit['Gender'] = Credit['Gender'].map({' Male': 1, 'Female': 0})
Credit['Student'] = Credit['Student'].map({'Yes': 1, 'No': 0})
Credit['Married'] = Credit['Married'].map({'Yes': 1, 'No': 0})
Credit['Ethnicity'] = Credit['Ethnicity'].map({"Asian": 2, 'Caucasian': 1, 'African American': 0})

n_features = 10  # Number of predictors
feature_names = Credit.columns[:-1].tolist()

# ---------------------------------------------------
# 2. Ridge Regression
# ---------------------------------------------------

# Define response and predictors
y = Credit['Balance'].values
X_features = Credit.drop('Balance', axis=1).values
n_samples=len(y)

# Standardize the predictors
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Define the lambda grid (equivalent to alpha in sklearn's Ridge)
# In sklearn, alpha = lambda
lambda_grid = 10 ** np.linspace(4, -2, 100)

# Initialize an array to store coefficients for each lambda
coef_matrix = np.zeros((len(lambda_grid), X_features.shape[1]))

# Fit Ridge Regression models for each lambda and store the coefficients
for i, lam in enumerate(lambda_grid):
    ridge = Ridge(alpha=lam, fit_intercept=True)
    ridge.fit(X_scaled, y)
    coef_matrix[i, :] = ridge.coef_

# Extract coefficients for the 50th and 60th models
# Note: Python uses 0-based indexing
index_50 = 49
index_60 = 59

coef_50 = coef_matrix[index_50, :]
coef_60 = coef_matrix[index_60, :]

print(f"\nLambda at 50th index: {lambda_grid[index_50]:.4f}")
print(f"Coefficients at 50th lambda:\n{dict(zip(feature_names, coef_50))}")
print(f"L2 norm of coefficients at 50th lambda: {np.linalg.norm(coef_50):.4f}")

print(f"\nLambda at 60th index: {lambda_grid[index_60]:.4f}")
print(f"Coefficients at 60th lambda:\n{dict(zip(feature_names, coef_60))}")
print(f"L2 norm of coefficients at 60th lambda: {np.linalg.norm(coef_60):.4f}")

# ---------------------------------------------------
# 3. Visualization of Coefficient Norms and Solution Path
# ---------------------------------------------------

# Calculate L2 norms of coefficients for all lambda values
l2_norms = np.linalg.norm(coef_matrix, axis=1)

# Plot L2 norm of coefficients vs lambda index
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(lambda_grid) + 1), l2_norms, marker='o', color='red', linewidth=2)
plt.xlabel('Lambda Index')
plt.ylabel('L2 Norm of Coefficients')
plt.title('Ridge Regression: L2 Norm of Coefficients vs Lambda Index')
plt.grid(True)
plt.show()

# Plot solution path for selected features
# Adjust the feature indices based on your dataset. Here, we'll select features X3, X4, X5, X10
# which correspond to indices 2, 3, 4, 9 (0-based indexing)
selected_features = feature_names
selected_indices = [feature_names.index(feat) for feat in selected_features]

plt.figure(figsize=(10, 6))
for idx, feature in zip(selected_indices, selected_features):
    plt.plot(range(1, len(lambda_grid) + 1), coef_matrix[:, idx], label=feature, linewidth=2)

plt.xlabel('Lambda Index')
plt.ylabel('Coefficient Value')
plt.title('Ridge Regression: Solution Path')
plt.legend()
plt.grid(True)
plt.show()

# ---------------------------------------------------
# 4. Prediction and Evaluation
# ---------------------------------------------------

# Split the data into training and testing sets
# Using 50% of the data for training as in the R code
np.random.seed(1)
train_indices = np.random.choice(range(n_samples), size=n_samples // 2, replace=False)
test_indices = np.array(list(set(range(n_samples)) - set(train_indices)))

X_train, X_test = X_scaled[train_indices, :], X_scaled[test_indices, :]
y_train, y_test = y[train_indices], y[test_indices]

# Fit Ridge Regression on training data with the predefined lambda grid
ridge_models = []
for lam in lambda_grid:
    ridge = Ridge(alpha=lam, fit_intercept=True)
    ridge.fit(X_train, y_train)
    ridge_models.append(ridge)

# Predict on test data with lambda=4 (find the closest lambda in the grid)
lambda_4 = 4
closest_lambda_4 = min(lambda_grid, key=lambda x: abs(x - lambda_4))
index_lambda_4 = np.where(lambda_grid == closest_lambda_4)[0][0]
ridge_pred_4 = ridge_models[index_lambda_4].predict(X_test)
mse_lambda_4 = mean_squared_error(y_test, ridge_pred_4)
print(f"\n10-fold CV Error with lambda=4: {mse_lambda_4:.4f}")

# Predict on test data with lambda=1e10 (effectively zero coefficients)
lambda_1e10 = 1e10
# Since our grid goes from 10^4 to 10^-2, 1e10 is beyond the grid. We'll take the largest lambda in the grid.
closest_lambda_1e10 = min(lambda_grid, key=lambda x: abs(x - lambda_1e10))
index_lambda_1e10 = np.where(lambda_grid == closest_lambda_1e10)[0][0]
ridge_pred_1e10 = ridge_models[index_lambda_1e10].predict(X_test)
mse_lambda_1e10 = mean_squared_error(y_test, ridge_pred_1e10)
print(f"10-fold CV Error with lambda=1e10 (using closest lambda={closest_lambda_1e10:.4f}): {mse_lambda_1e10:.4f}")

# ---------------------------------------------------
# 5. Cross-Validation to Choose the Best Lambda
# ---------------------------------------------------

from sklearn.linear_model import RidgeCV

# Define cross-validation with the same lambda grid
ridge_cv = RidgeCV(alphas=lambda_grid, cv=10)
ridge_cv.fit(X_train, y_train)

# Best lambda that minimizes CV error
best_lambda = ridge_cv.alpha_
print(f"\nBest lambda from cross-validation: {best_lambda:.4f}")

# Predict on test data using the best lambda
ridge_best = Ridge(alpha=best_lambda, fit_intercept=True)
ridge_best.fit(X_train, y_train)
ridge_pred_best = ridge_best.predict(X_test)
mse_best_lambda = mean_squared_error(y_test, ridge_pred_best)
print(f"10-fold CV Error with best lambda={best_lambda:.4f}: {mse_best_lambda:.4f}")

# ---------------------------------------------------
# 6. Fit Ridge Regression on the Full Dataset with Best Lambda
# ---------------------------------------------------

ridge_full = Ridge(alpha=best_lambda, fit_intercept=True)
ridge_full.fit(X_scaled, y)
full_coefficients = ridge_full.coef_

print(f"\nRidge Regression Coefficients on Full Dataset with lambda={best_lambda:.4f}:")
print(dict(zip(feature_names, full_coefficients)))

# Optionally, include the intercept
print(f"Intercept: {ridge_full.intercept_:.4f}")

In [ ]:
############### KNN regression for best subset selection
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.model_selection import cross_val_score, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

def evaluate_subset(subset, X, y, model, scoring, cv_folds, random_state):
    X_subset = X[list(subset)]
    # Perform cross-validation
    scores = cross_val_score(model, X_subset, y, cv=KFold(n_splits=cv_folds, shuffle=True, random_state=random_state), scoring=scoring)
    mean_score = np.mean(scores)
    return (subset, -mean_score)

def best_subset_selection_parallel(X, y, model, scoring='neg_mean_squared_error', cv_folds=5, n_jobs=-1, random_state=42):
    feature_names = X.columns
    n_features = len(feature_names)
    results = []
    for k in range(1, n_features + 1):
        print(f"Evaluating {k}-feature subsets...")
        subsets = list(combinations(feature_names, k))
        subset_scores = Parallel(n_jobs=n_jobs)(delayed(evaluate_subset)(subset, X, y, model, scoring, cv_folds, random_state) for subset in subsets)
        results.extend(subset_scores)
    
    return results

def main():
    # Load the Dataset
    Credit = pd.read_csv('Credit.csv')
    Credit = Credit.drop('ID', axis=1)
    print(Credit.head(10))
    [num_samples,num_predictors]=Credit.shape
    Credit['Gender'] = Credit['Gender'].map({' Male': 1, 'Female': 0})
    Credit['Student'] = Credit['Student'].map({'Yes': 1, 'No': 0})
    Credit['Married'] = Credit['Married'].map({'Yes': 1, 'No': 0})
    Credit['Ethnicity'] = Credit['Ethnicity'].map({"Asian": 2, 'Caucasian': 1, 'African American': 0})
    X = Credit.drop('Balance', axis=1)
    y = Credit['Balance']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

    knn = KNeighborsRegressor(n_neighbors=10)
    print("Starting Best Subset Selection with KNN Regression (Parallel)...")
    results = best_subset_selection_parallel(X_scaled, y, model=knn, scoring='neg_mean_squared_error', cv_folds=5, n_jobs=-1, random_state=42)
    print("Best Subset Selection Completed.")
    results_df = pd.DataFrame(results, columns=['Features', 'CV_Score'])
    results_df = results_df.sort_values(by='CV_Score', ascending=True).reset_index(drop=True)

    print("\nTop 20 Best Feature Subsets:")
    print(results_df.head(20))
    best_features = results_df.iloc[0]['Features']
    print(f"\nBest feature subset: {best_features}")
    X_best = X_scaled[list(best_features)]
    final_scores = cross_val_score(knn, X_best, y, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring='neg_mean_squared_error')
    final_rmse = np.sqrt(-final_scores)

    print(f"\nCross-Validated RMSE of the best model: {final_rmse}")
    print(f"Average RMSE: {final_rmse.mean():.4f}")
    print(f"Std of RMSE: {final_rmse.std():.4f}")

if __name__ == "__main__":
    main()

In [ ]:
############### KNN regression for best subset selection with noisy variables
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.model_selection import cross_val_score, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

def evaluate_subset(subset, X, y, model, scoring, cv_folds, random_state):
    X_subset = X[list(subset)]
    # Perform cross-validation
    scores = cross_val_score(model, X_subset, y, cv=KFold(n_splits=cv_folds, shuffle=True, random_state=random_state), scoring=scoring)
    mean_score = np.mean(scores)
    return (subset, -mean_score)

def best_subset_selection_parallel(X, y, model, scoring='neg_mean_squared_error', cv_folds=5, n_jobs=-1, random_state=42):
    feature_names = X.columns
    #n_features = len(feature_names)  ######## maximum model size
    n_features=4
    results = []
    for k in range(1, n_features + 1):
        print(f"Evaluating {k}-feature subsets...")
        subsets = list(combinations(feature_names, k))
        subset_scores = Parallel(n_jobs=n_jobs)(delayed(evaluate_subset)(subset, X, y, model, scoring, cv_folds, random_state) for subset in subsets)
        results.extend(subset_scores)
    
    return results

def main():
    # Load the Dataset
    Credit = pd.read_csv('Credit.csv')
    Credit = Credit.drop('ID', axis=1)
    [num_samples,num_predictors]=Credit.shape
    Credit['Gender'] = Credit['Gender'].map({' Male': 1, 'Female': 0})
    Credit['Student'] = Credit['Student'].map({'Yes': 1, 'No': 0})
    Credit['Married'] = Credit['Married'].map({'Yes': 1, 'No': 0})
    Credit['Ethnicity'] = Credit['Ethnicity'].map({"Asian": 2, 'Caucasian': 1, 'African American': 0})
    ###### create noisy columns
    column_names = [f'Noisy{i}' for i in range(1, 20 + 1)]
    data = {col: np.random.normal(loc=0, scale=1, size=400) for col in column_names}
    Credit=pd.concat([Credit, pd.DataFrame(data)], axis=1, sort=False)
    print(Credit.head(4))
    
    X = Credit.drop('Balance', axis=1)
    y = Credit['Balance']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

    knn = KNeighborsRegressor(n_neighbors=10)
    print("Starting Best Subset Selection with KNN Regression (Parallel)...")
    results = best_subset_selection_parallel(X_scaled, y, model=knn, scoring='neg_mean_squared_error', cv_folds=5, n_jobs=-1, random_state=42)
    print("Best Subset Selection Completed.")
    results_df = pd.DataFrame(results, columns=['Features', 'CV_Score'])
    results_df = results_df.sort_values(by='CV_Score', ascending=True).reset_index(drop=True)

    print("\nTop 20 Best Feature Subsets:")
    print(results_df.head(20))
    best_features = results_df.iloc[0]['Features']
    print(f"\nBest feature subset: {best_features}")
    X_best = X_scaled[list(best_features)]
    final_scores = cross_val_score(knn, X_best, y, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring='neg_mean_squared_error')
    final_rmse = np.sqrt(-final_scores)

    print(f"\nCross-Validated RMSE of the best model: {final_rmse}")
    print(f"Average RMSE: {final_rmse.mean():.4f}")
    print(f"Std of RMSE: {final_rmse.std():.4f}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm

# ---------------------------
# 1. Load the Wage Dataset
# ---------------------------

# Load the Wage dataset from ISLR's GitHub repository
Wage = pd.read_csv('Wage.csv')

# Display the first few rows
print("First five rows of the Wage dataset:")
print(Wage.head())

# ---------------------------
# 2. Plotting Wage versus Age
# ---------------------------

# Plot Wage versus Age
plt.figure(figsize=(8,6))
plt.scatter(Wage['age'], Wage['wage'], color='darkgrey', alpha=0.5)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Wage', fontsize=12)
plt.title('Wage versus Age', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# ---------------------------
# 3. Polynomial Regression and Step Functions
# ---------------------------


# ---------------------------
# b. Fit Degree-4 Polynomial (Raw Terms)
# ---------------------------

# Fit a degree-4 polynomial using raw polynomial terms
fit_poly4_raw = smf.ols('wage ~ age + I(age**2) + I(age**3) + I(age**4)', data=Wage).fit()

# Display the summary of the fitted model
print("\nSummary of Degree-4 Polynomial Regression (Raw Terms):")
print(fit_poly4_raw.summary())

# ---------------------------
# c. Compact Polynomial Regression with cbind-like Method
# ---------------------------

# Fit a degree-4 polynomial using a compact formula (equivalent to cbind in R)
fit_poly4_compact = smf.ols('wage ~ age + I(age**2) + I(age**3) + I(age**4)', data=Wage).fit()

# Display the summary of the fitted model
print("\nSummary of Degree-4 Polynomial Regression (Compact Method):")
print(fit_poly4_compact.summary())


# ---------------------------
# e. Predictions and Confidence Bands for Degree-4 Polynomial (Raw)
# ---------------------------

# Predict using the degree-4 raw polynomial model
preds_poly4_raw = fit_poly4_raw.get_prediction(pred_data)
preds_summary_poly4_raw = preds_poly4_raw.summary_frame()

# Confidence bands
ci_upper_poly4_raw = preds_summary_poly4_raw['mean_ci_upper']
ci_lower_poly4_raw = preds_summary_poly4_raw['mean_ci_lower']
pred_fit_poly4_raw = preds_summary_poly4_raw['mean']

# ---------------------------
# f. Plotting Degree-4 Polynomial Fits with Confidence Bands
# ---------------------------

# Create subplots for Degree-4 Orthogonal and Raw Polynomial Fits
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)


# Plot for Degree-4 Raw Polynomial
axes[1].scatter(Wage['age'], Wage['wage'], color='darkgrey', s=10, alpha=0.5, label='Data')
axes[1].plot(age_grid, pred_fit_poly4_raw, color='red', linewidth=2, label='Degree-4 Raw')
axes[1].fill_between(age_grid, ci_lower_poly4_raw, ci_upper_poly4_raw, color='red', alpha=0.2, label='95% Confidence Interval')
axes[1].set_xlabel('Age', fontsize=12)
axes[1].set_title('Degree-4 Raw Polynomial Fit', fontsize=14)
axes[1].legend()

# Add an overall title
plt.suptitle('Polynomial Regression: Degree-4 Orthogonal vs Raw Polynomials', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to accommodate the suptitle
plt.show()

In [ ]:
####### polynomial logistic regression

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm


import numpy as np

def poly(x, degree):
    return np.column_stack([x**i for i in range(1, degree + 1)])


# ---------------------------
# 1. Load the Wage Dataset
# ---------------------------

# Load the Wage dataset from ISLR's GitHub repository
Wage = pd.read_csv('Wage.csv')

# ---------------------------
# 2. Plotting Wage versus Age
# ---------------------------

# Define the range for age grid based on the data
agelims = Wage['age'].min(), Wage['age'].max()
age_grid = np.linspace(agelims[0], agelims[1], 1000)

# Plot Wage versus Age
plt.figure(figsize=(10, 6))
plt.scatter(Wage['age'], Wage['wage'], color='darkgrey', alpha=0.5)
plt.xlabel('Age', fontsize=14)
plt.ylabel('Wage', fontsize=14)
plt.title('Wage versus Age', fontsize=16)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# ---------------------------
# 3. Fit a Polynomial Logistic Regression Model
# ---------------------------

# Define the threshold for classification
threshold = 250  # Adjust based on the data

# Create a binary target variable: 1 if wage > threshold, else 0
Wage['wage_binary'] = (Wage['wage'] > threshold).astype(int)

# Fit a 4th-degree polynomial logistic regression model 
fit = smf.glm(formula='wage_binary ~ poly(age,4)', data=Wage, family=sm.families.Binomial()).fit()

# Display the summary of the fitted model
print("\nSummary of Polynomial Logistic Regression Model (Degree 4):")
print(fit.summary())

# ---------------------------
# 4. Generate Predictions and Confidence Intervals
# ---------------------------

# Generate predictions on the age grid
preds = fit.get_prediction(pd.DataFrame({'age': age_grid}))
preds_summary = preds.summary_frame()

# Extract predicted probabilities
pfit = preds_summary['mean']

# Extract confidence intervals on the logit scale
ci_upper_logit = preds_summary['mean_ci_upper']
ci_lower_logit = preds_summary['mean_ci_lower']

# Convert logit confidence intervals to probability scale using the logistic function
se_bands_logit = np.vstack((ci_upper_logit, ci_lower_logit)).T
se_bands = np.exp(se_bands_logit) / (1 + np.exp(se_bands_logit))

# ---------------------------
# 5. Plot the Fitted Probabilities with Confidence Bands
# ---------------------------

plt.figure(figsize=(10, 6))
plt.xlim(agelims)
plt.ylim(0, 0.2)  # Adjust based on the threshold and data

# Scatter plot of observations with jittered x-values
plt.scatter(Wage['age'] + np.random.uniform(-0.1, 0.1, size=Wage.shape[0]),
            Wage['wage_binary'] / 5,  # Scaling to fit y-axis limits
            marker='|',
            color='darkgrey',
            alpha=0.5,
            label='Data Points')

# Plot the fitted probability curve
plt.plot(age_grid, pfit, color='blue', linewidth=2, label='Fitted Probability')

# Plot the confidence bands
plt.fill_between(age_grid, se_bands[:,1], se_bands[:,0],
                 color='blue', alpha=0.2, label='95% Confidence Interval')

# Add labels and title
plt.xlabel('Age', fontsize=14)
plt.ylabel('Probability (wage > $250)', fontsize=14)
plt.title('Polynomial Logistic Regression: Wage > $250 vs Age', fontsize=16)

# Add legend
plt.legend()

# Add grid
plt.grid(True, linestyle='--', alpha=0.5)

# Show the plot
plt.show()


fit_degree5 = smf.glm(formula='wage_binary ~ poly(age, 5)', data=Wage, family=sm.families.Binomial()).fit()


# ---------------------------
# 7. Plotting the Degree-5 Polynomial Fit with Confidence Bands
# ---------------------------

# Predict using the degree-5 polynomial model
preds_poly5 = fit_degree5.get_prediction(pd.DataFrame({'age': age_grid}))
preds_summary_poly5 = preds_poly5.summary_frame()

# Confidence bands
ci_upper_poly5 = preds_summary_poly5['mean_ci_upper']
ci_lower_poly5 = preds_summary_poly5['mean_ci_lower']
pred_fit_poly5 = preds_summary_poly5['mean']

# Plot the Degree-5 Polynomial Fit
plt.figure(figsize=(10, 6))
plt.scatter(Wage['age'], Wage['wage_binary'], color='darkgrey', s=10, alpha=0.5, label='Data Points')
plt.plot(age_grid, pred_fit_poly5, color='green', linewidth=2, label='Degree-5 Polynomial')
plt.fill_between(age_grid, ci_lower_poly5, ci_upper_poly5, color='green', alpha=0.2, label='95% Confidence Interval')
plt.xlabel('Age', fontsize=14)
plt.ylabel('Probability (wage > $250)', fontsize=14)
plt.title('Degree-5 Polynomial Logistic Regression', fontsize=16)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Print coefficients summary for Degree-5 polynomial
print("\nSummary of Degree-5 Polynomial Logistic Regression:")
print(fit_degree5.summary())

In [ ]:
######### step function and logistic regression
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Suppress scientific notation in pandas
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# ---------------------------
# 1. Load the Wage Dataset
# ---------------------------

# Load the Wage dataset from ISLR's GitHub repository
Wage = pd.read_csv('Wage.csv')
df=Wage
# Create binary outcome
df['high_wage'] = (df['wage'] > 250).astype(int)

# Create age grids
age_min, age_max = df['age'].min(), df['age'].max()
age_grid = np.linspace(age_min, age_max, 100)

# ----------------------------------
# 1. Plot Wage vs Age
# ----------------------------------
plt.figure(figsize=(8, 6))
plt.scatter(df['age'], df['wage'], color='darkgrey', alpha=0.5)
plt.title('Wage versus Age')
plt.xlabel('Age')
plt.ylabel('Wage')
plt.grid(True)
plt.show()

# ----------------------------------
# 2. Linear Regression with 4 Bins
# ----------------------------------
df['age_cut_4'] = pd.cut(df['age'], bins=4)
print("Frequency table for age cut into 4 bins:")
print(df['age_cut_4'].value_counts())

model_lm_4 = smf.ols('wage ~ C(age_cut_4)', data=df).fit()
print("\nLinear Regression Coefficients (age cut into 4 bins):")
print(model_lm_4.summary().tables[1])

# Prediction
pred_df_4 = pd.DataFrame({'age': age_grid})
pred_df_4['age_cut_4'] = pd.cut(pred_df_4['age'], bins=4)

predictions_4 = model_lm_4.get_prediction(pred_df_4)
pred_summary_4 = predictions_4.summary_frame(alpha=0.05)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(df['age'], df['wage'], color='darkgrey', alpha=0.5, label='Data')
plt.plot(age_grid, pred_summary_4['mean'], color='blue', label='Fitted Step Function')
plt.fill_between(age_grid, pred_summary_4['mean_ci_lower'], pred_summary_4['mean_ci_upper'],
                 color='red', alpha=0.3, label='95% Confidence Bands')
plt.title('Linear Regression with Step Function (4 Bins)')
plt.xlabel('Age')
plt.ylabel('Wage')
plt.legend()
plt.grid(True)
plt.show()

# ----------------------------------
# 3. Linear Regression with 8 Bins
# ----------------------------------
df['age_cut_8'] = pd.cut(df['age'], bins=8)
print("Frequency table for age cut into 8 bins:")
print(df['age_cut_8'].value_counts())

model_lm_8 = smf.ols('wage ~ C(age_cut_8)', data=df).fit()
print("\nLinear Regression Coefficients (age cut into 8 bins):")
print(model_lm_8.summary().tables[1])

# Prediction
pred_df_8 = pd.DataFrame({'age': age_grid})
pred_df_8['age_cut_8'] = pd.cut(pred_df_8['age'], bins=8)

predictions_8 = model_lm_8.get_prediction(pred_df_8)
pred_summary_8 = predictions_8.summary_frame(alpha=0.05)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(df['age'], df['wage'], color='darkgrey', alpha=0.5, label='Data')
plt.plot(age_grid, pred_summary_8['mean'], color='blue', label='Fitted Step Function')
plt.fill_between(age_grid, pred_summary_8['mean_ci_lower'], pred_summary_8['mean_ci_upper'],
                 color='red', alpha=0.3, label='95% Confidence Bands')
plt.title('Linear Regression with Step Function (8 Bins)')
plt.xlabel('Age')
plt.ylabel('Wage')
plt.legend()
plt.grid(True)
plt.show()



#############
#4. Logistic regression with 4 Bins
##

# Load the Wage dataset from ISLR's GitHub repository
wage = pd.read_csv('Wage.csv')

# 3. Fit a step function logistic regression model
# Create a binary target variable: 1 if wage > 250, else 0
wage['high_wage'] = (wage['wage'] > 250).astype(int)

# Define the number of bins for the step function
num_bins = 4
wage['age_bin'] = pd.cut(wage['age'], bins=num_bins)

# Fit logistic regression using statsmodels with categorical age bins
model = smf.glm(
    formula='high_wage ~ C(age_bin)',
    data=wage,
    family=sm.families.Binomial()
).fit()

print(model.summary())

# 4. Make predictions on a grid of age values
# Define age limits
agelims = [wage['age'].min(), wage['age'].max()]
age_grid = np.linspace(agelims[0], agelims[1], 100)

# Create a DataFrame for prediction by cutting the age_grid into the same bins
age_bins = pd.cut(age_grid, bins=num_bins)
pred_data = pd.DataFrame({'age_bin': age_bins})

# Get predictions on the logit scale with standard errors
predictions = model.get_prediction(pred_data)
pred_summary = predictions.summary_frame()

# Calculate pfit and confidence bands on the probability scale
pfit = pred_summary['mean']  # Predicted probabilities
se_bands_logit_upper = pred_summary['mean_ci_upper']
se_bands_logit_lower = pred_summary['mean_ci_lower']

# Alternatively, calculate confidence intervals manually on the logit scale
# and then transform them to the probability scale
# However, statsmodels already provides the confidence intervals on the probability scale

# 5. Plot the confidence bands
plt.figure(figsize=(10, 6))

# Plot the binary outcome with some jitter
# Adding jitter by adding small random noise to the age
jitter_strength = 0.3
jittered_age = wage['age'] + np.random.uniform(-jitter_strength, jitter_strength, size=wage.shape[0])
plt.scatter(jittered_age, wage['high_wage'], 
            color='darkgrey', alpha=0.5, marker='|', label='Data')

# Plot the predicted probabilities
plt.plot(age_grid, pfit, color='blue', lw=2, label='Fitted Probability')

# Plot the confidence bands
plt.fill_between(age_grid, pred_summary['mean_ci_lower'], pred_summary['mean_ci_upper'], 
                 color='red', alpha=0.3, label='95% Confidence Band')

# Customize the plot
plt.xlim(agelims)
plt.ylim(-0.05, 1.05)
plt.xlabel('Age')
plt.ylabel('Probability of Wage > 250')
plt.title('Step Function Logistic Model')
plt.legend()
plt.show()






####
# Logistci regression with Step functions 8bins
####
# Load the Wage dataset from ISLR's GitHub repository
wage = pd.read_csv('Wage.csv')

# 3. Fit a step function logistic regression model
# Create a binary target variable: 1 if wage > 250, else 0
wage['high_wage'] = (wage['wage'] > 250).astype(int)

# Define the number of bins for the step function
num_bins = 10
wage['age_bin'] = pd.cut(wage['age'], bins=num_bins)

# Fit logistic regression using statsmodels with categorical age bins
model = smf.glm(
    formula='high_wage ~ C(age_bin)',
    data=wage,
    family=sm.families.Binomial()
).fit()

print(model.summary())

# 4. Make predictions on a grid of age values
# Define age limits
agelims = [wage['age'].min(), wage['age'].max()]
age_grid = np.linspace(agelims[0], agelims[1], 100)

# Create a DataFrame for prediction by cutting the age_grid into the same bins
age_bins = pd.cut(age_grid, bins=num_bins)
pred_data = pd.DataFrame({'age_bin': age_bins})

# Get predictions on the logit scale with standard errors
predictions = model.get_prediction(pred_data)
pred_summary = predictions.summary_frame()

# Calculate pfit and confidence bands on the probability scale
pfit = pred_summary['mean']  # Predicted probabilities
se_bands_logit_upper = pred_summary['mean_ci_upper']
se_bands_logit_lower = pred_summary['mean_ci_lower']

# Alternatively, calculate confidence intervals manually on the logit scale
# and then transform them to the probability scale
# However, statsmodels already provides the confidence intervals on the probability scale

# 5. Plot the confidence bands
plt.figure(figsize=(10, 6))

# Plot the binary outcome with some jitter
# Adding jitter by adding small random noise to the age
jitter_strength = 0.3
jittered_age = wage['age'] + np.random.uniform(-jitter_strength, jitter_strength, size=wage.shape[0])
plt.scatter(jittered_age, wage['high_wage'], 
            color='darkgrey', alpha=0.5, marker='|', label='Data')

# Plot the predicted probabilities
plt.plot(age_grid, pfit, color='blue', lw=2, label='Fitted Probability')

# Plot the confidence bands
plt.fill_between(age_grid, pred_summary['mean_ci_lower'], pred_summary['mean_ci_upper'], 
                 color='red', alpha=0.3, label='95% Confidence Band')

# Customize the plot
plt.xlim(agelims)
plt.ylim(-0.05, 1.05)
plt.xlabel('Age')
plt.ylabel('Probability of Wage > 250')
plt.title('Step Function Logistic Model')
plt.legend()
plt.show()


